 # # EDA — Restaurant Survival Classification

 #

 **Machine Learning 1 — Prof. Piotr Wójcik — a.y. 2025/2026**



 Objective: exploratory analysis of the `restaurants_train.csv` dataset

 to validate the Workflow Strategy v2.0 hypotheses and make informed

 decisions about imputation, encoding, and feature engineering.



In [ ]:
# === IMPORTS ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

pd.set_option('display.max_columns', 90)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)



In [ ]:
# === 1. DATA LOADING ===
train = pd.read_csv('Data/restaurants_train.csv')
test  = pd.read_csv('Data/restaurants_test.csv')

print(f"Train shape: {train.shape}")
print(f"Test  shape: {test.shape}")
print(f"\nColumns in train but not in test: {set(train.columns) - set(test.columns)}")



In [ ]:
# First look at the structure
train.info()



In [ ]:
train.head()



In [ ]:
train.describe()



  # --- 2. TARGET — Imbalance of `status_closed`



  Knowing the exact proportion tells us how aggressive



  the `class_weight='balanced'` needs to be.

In [ ]:
target_counts = train['status_closed'].value_counts()
target_pct    = train['status_closed'].value_counts(normalize=True) * 100

print("Absolute counts:")
print(target_counts)
print(f"\nProportions:")
print(target_pct.round(2))
print(f"\nRatio Class 0 / Class 1: {target_counts[0] / target_counts[1]:.2f}")



In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
target_counts.plot(kind='bar', color=['#2ecc71', '#e74c3c'], edgecolor='black', ax=ax)
ax.set_title('Distribution of status_closed')
ax.set_xlabel('status_closed')
ax.set_ylabel('Count')
for i, v in enumerate(target_counts):
    ax.text(i, v + 200, f'{v}\n({target_pct.iloc[i]:.1f}%)', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()



  # --- 3. PROBLEMATIC VARIABLES — Unexpected types and values





   3a. `weekends_only` and `workdays_only` (String, not Boolean)

In [ ]:
print("=== weekends_only ===")
print(f"Dtype: {train['weekends_only'].dtype}")
print(train['weekends_only'].value_counts(dropna=False))
print()
print("=== workdays_only ===")
print(f"Dtype: {train['workdays_only'].dtype}")
print(train['workdays_only'].value_counts(dropna=False))



  3b. `price_level` — Unique values (expected: 1-4 categorical)

In [ ]:
print(f"Dtype: {train['price_level'].dtype}")
print(f"\nUnique values: {sorted(train['price_level'].dropna().unique())}")
print(f"\nDistribution (including NaNs):")
print(train['price_level'].value_counts(dropna=False).sort_index())



  3c. `first_review_year_max` — Anomaly check

In [ ]:
print(f"Dtype: {train['first_review_year_max'].dtype}")
print(f"Min: {train['first_review_year_max'].min()}, Max: {train['first_review_year_max'].max()}")
print(f"NaN: {train['first_review_year_max'].isna().sum()}")
print(f"\nDistribution:")
print(train['first_review_year_max'].value_counts(dropna=False).sort_index())



  # --- 4. CARDINALITY OF `category_top20`

In [ ]:
cat_counts = train['category_top20'].value_counts(dropna=False)
cat_pct    = train['category_top20'].value_counts(dropna=False, normalize=True) * 100

cat_summary = pd.DataFrame({
    'count': cat_counts,
    'pct': cat_pct.round(2)
})
print(f"Number of unique categories: {train['category_top20'].nunique()}")
print(f"\nFull distribution:")
print(cat_summary)



In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
cat_counts.plot(kind='bar', edgecolor='black', ax=ax)
ax.set_title('Distribution of category_top20')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()



  # --- 5. MISSING VALUES — Patterns and concentration

In [ ]:
# Count and percentage of NaNs per column
missing_count = train.isnull().sum()
missing_pct   = (train.isnull().sum() / len(train) * 100).round(2)

missing_df = pd.DataFrame({
    'n_missing': missing_count,
    'pct_missing': missing_pct
}).query('n_missing > 0').sort_values('pct_missing', ascending=False)

print(f"Columns with at least 1 NaN: {len(missing_df)} out of {train.shape[1]}")
print(f"\n{missing_df}")




In [ ]:
# Visualization with missingno — matrix
msno.matrix(train, figsize=(16, 8), fontsize=8)
plt.title('Missing Values — Matrix')
plt.tight_layout()
plt.show()



In [ ]:
# Hypothesis check: do NaNs in rating variables coincide
# with user_ratings_total == 0?

rating_vars = ['rating_avg', 'rating_std', 'review_length_avg',
               'ratings_avg_12m_prior', 'ratings_num_12m_prior']

# How many places have 0 total reviews?
zero_ratings = (train['user_ratings_total'] == 0).sum()
print(f"Places with user_ratings_total == 0: {zero_ratings} ({zero_ratings/len(train)*100:.2f}%)")

# For each rating variable, how many NaNs coincide with user_ratings_total == 0?
print(f"\nNaN ↔ user_ratings_total == 0 Overlap:")
for var in rating_vars:
    nan_mask = train[var].isna()
    overlap  = (nan_mask & (train['user_ratings_total'] == 0)).sum()
    total_nan = nan_mask.sum()
    if total_nan > 0:
        print(f"  {var}: {total_nan} total NaNs, {overlap} coincide with ratings=0 ({overlap/total_nan*100:.1f}%)")
    else:
        print(f"  {var}: no NaNs")



In [ ]:
# Select only numeric columns
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
# Remove restaurant_id and the target
numeric_cols = [c for c in numeric_cols if c not in ['restaurant_id', 'status_closed']]

corr_matrix = train[numeric_cols].corr(method='spearman')



In [ ]:
# Extract pairs with |ρ| > 0.90
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        rho = corr_matrix.iloc[i, j]
        if abs(rho) > 0.90:
            high_corr_pairs.append({
                'var_1': corr_matrix.columns[i],
                'var_2': corr_matrix.columns[j],
                'spearman_rho': round(rho, 4)
            })

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('spearman_rho',
                                                           key=abs,
                                                           ascending=False)
print(f"Pairs with |Spearman ρ| > 0.90: {len(high_corr_df)}")
print()
print(high_corr_df.to_string(index=False))



In [ ]:
# Correlation matrix heatmap (compact overview)
fig, ax = plt.subplots(figsize=(18, 15))
sns.heatmap(corr_matrix, cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.3,
            xticklabels=True, yticklabels=True, ax=ax)
ax.set_title('Spearman Correlation Matrix — Numeric Variables')
plt.xticks(fontsize=6, rotation=90)
plt.yticks(fontsize=6)
plt.tight_layout()
plt.show()



  # --- 6. DISTRIBUTIONS — Tails and skewness of count variables



  Objective: decide whether PowerTransformer or RobustScaler are needed.

In [ ]:
count_vars = ['user_ratings_total', 'catch_restaurant_count_500m',
              'catch_restaurant_count_1000m', 'catch_restaurant_count_2000m',
              'poi_count_100m', 'poi_count_500m', 'poi_count_2000m',
              'residents', 'place_age_days', 'lang_pl_count']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.ravel()

for i, var in enumerate(count_vars):
    ax = axes[i]
    train[var].dropna().hist(bins=50, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(var, fontsize=9)
    skew_val = train[var].skew()
    ax.text(0.95, 0.95, f'skew={skew_val:.2f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Count variables distributions — Skewness', fontsize=13)
plt.tight_layout()
plt.show()



In [ ]:
# Tabular skewness for all numeric variables
skewness = train[numeric_cols].skew().sort_values(ascending=False)
print("Top 15 variables by skewness (most asymmetrical):")
print(skewness.head(15).round(3))
print(f"\n...and the 5 least asymmetrical:")
print(skewness.tail(5).round(3))



   # 02 — Data Preparation: Missing Values & Encoding



 **Machine Learning 1 — Prof. Piotr Wójcik — a.y. 2025/2026**

 This script applies the imputation and encoding strategy agreed upon in the Workflow Strategy v2.0 analysis.

 Everything done here is PRE-PIPELINE (deterministic transformations that do not depend on training set statistics, therefore no data leakage).

 Transformations depending on the training set (median, scaling)

 will be encapsulated in the Pipeline in the next step.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 90)
pd.set_option('display.float_format', '{:.4f}'.format)


In [ ]:
# === LOADING ===
train = pd.read_csv('Data/restaurants_train.csv')
test  = pd.read_csv('Data/restaurants_test.csv')

# Save target and id before any transformation
target = train['status_closed'].copy()
train_id = train['restaurant_id'].copy()
test_id  = test['restaurant_id'].copy()

# Merge train and test to apply deterministic transformations
# consistently. We add a flag to separate them later.
train['_is_train'] = 1
test['_is_train']  = 0
# Add status_closed to test as NaN to be able to concatenate
test['status_closed'] = np.nan

df = pd.concat([train, test], axis=0, ignore_index=True)
print(f"Combined dataset: {df.shape}")


   ---







   ## GROUP 0 — Synthetic indicator `is_new_restaurant`







   Single proxy for all physiological missings.







   Captures the signal "this place has no review activity".

In [ ]:
df['is_new_restaurant'] = (df['user_ratings_total'] == 0).astype(int)
print(f"'New' places (user_ratings_total == 0): {df['is_new_restaurant'].sum()} "
      f"({df['is_new_restaurant'].mean()*100:.2f}%)")




   ---







   ## GROUP 1 — Physiological NaNs → imputation to 0







   Variables where NaN = "no reviews / place too recent".







   0 is semantically correct + `is_new_restaurant` signals to the model







   that these zeros are not real zeros.

In [ ]:
# Rating and count variables where NaN = lack of activity
group1_zero_impute = [
    # Star ratings
    'rating_5', 'rating_4', 'rating_3', 'rating_2', 'rating_1',
    'rating_avg', 'rating_std',
    # Temporal review counts
    'ratings_num_1m_prior', 'ratings_num_3m_prior',
    'ratings_num_6m_prior', 'ratings_num_9m_prior',
    'ratings_num_12m_prior',
    # Linguistic variables (counts and shares)
    'lang_pl_count', 'rating_pl', 'rating_foreign',
    'foreign_lang_share',
    # Review text
    'review_has_text_pct', 'review_length_avg', 'review_length_std',
]

before_na = df[group1_zero_impute].isna().sum().sum()
df[group1_zero_impute] = df[group1_zero_impute].fillna(0)
after_na = df[group1_zero_impute].isna().sum().sum()
print(f"Group 1 — NaNs imputed to 0: {before_na - after_na}")




   ---







   ## GROUP 2 — NaNs to be imputed with MEDIAN (inside Pipeline)







   These variables remain with NaNs for now. They will be handled by







   SimpleImputer(strategy='median') in the Pipeline.















   Includes:







   - Hours block: hours_open, hours_open_weekends, hours_open_workdays,







     days_evenings_only, days_mornings_only







   - Temporal average ratings: ratings_avg_1m..12m_prior







   - Linguistic average ratings: rating_mean_pl, rating_mean_foreign,







     rating_mean_lang_ratio







   - Spatial (area average rating/age): catch_rating_avg_*, catch_place_age_days_*







   - place_age_days, restaurants_per_capita

In [ ]:
group2_median_pipeline = [
    'hours_open', 'hours_open_weekends', 'hours_open_workdays',
    'days_evenings_only', 'days_mornings_only',
    'ratings_avg_1m_prior', 'ratings_avg_3m_prior',
    'ratings_avg_6m_prior', 'ratings_avg_9m_prior',
    'ratings_avg_12m_prior',
    'rating_mean_pl', 'rating_mean_foreign', 'rating_mean_lang_ratio',
    'catch_rating_avg_500m', 'catch_place_age_days_500m',
    'catch_rating_avg_1000m', 'catch_place_age_days_1000m',
    'catch_rating_avg_2000m', 'catch_place_age_days_2000m',
    'place_age_days', 'first_review_year_max',
    'restaurants_per_capita',
]

print(f"Group 2 — Variables left with NaNs (median in Pipeline): {len(group2_median_pipeline)}")
print(f"Residual NaNs in these columns: {df[group2_median_pipeline].isna().sum().sum()}")




   ---







   ## GROUP 3 — `price_level` → OHE with "missing" category

In [ ]:
# Convert to string to create the "missing" category
df['price_level'] = df['price_level'].fillna('missing')
df['price_level'] = df['price_level'].replace({
    1.0: '1', 2.0: '2', 3.0: '3', 4.0: '4',
    '1.0': '1', '2.0': '2', '3.0': '3', '4.0': '4'
})

print("price_level after transformation:")
print(df['price_level'].value_counts().sort_index())

# OHE with drop_first=False (keep all dummies including "missing")
price_dummies = pd.get_dummies(df['price_level'], prefix='price_level', dtype=int)
df = pd.concat([df, price_dummies], axis=1)
df.drop(columns=['price_level'], inplace=True)

print(f"\nDummies created: {[c for c in df.columns if c.startswith('price_level_')]}")




   ---







   ## GROUP 4 — `tagcat_*` → imputation to 0







   NaN = Google has no tags for that category = 0 declared tags.







   `tagcat_payment_options` (71.6% NaN) is kept: LASSO will







   decide if it has a signal or not.

In [ ]:
group4_tagcat = [
    'tagcat_services', 'tagcat_amenities', 'tagcat_atmosphere',
    'tagcat_offerings', 'tagcat_social_inclusivity', 'tagcat_payment_options',
]

before_na = df[group4_tagcat].isna().sum().sum()
df[group4_tagcat] = df[group4_tagcat].fillna(0)
after_na = df[group4_tagcat].isna().sum().sum()
print(f"Group 4 — NaNs imputed to 0: {before_na - after_na}")




   ---







   ## GROUP 5 — Spatial variables `catch_restaurant_count_*` → 0







   NaN = no restaurant in the radius = 0 competitors.

In [ ]:
group5_catch_count = [
    'catch_restaurant_count_500m',
    'catch_restaurant_count_1000m',
    'catch_restaurant_count_2000m',
]

before_na = df[group5_catch_count].isna().sum().sum()
df[group5_catch_count] = df[group5_catch_count].fillna(0)
after_na = df[group5_catch_count].isna().sum().sum()
print(f"Group 5 — NaNs imputed to 0: {before_na - after_na}")




   ---







   ## GROUP 6 — `category_top20` → OHE with "missing" category

In [ ]:
df['category_top20'] = df['category_top20'].fillna('missing')

print(f"Unique categories (including 'missing'): {df['category_top20'].nunique()}")

cat_dummies = pd.get_dummies(df['category_top20'], prefix='cat', dtype=int)
df = pd.concat([df, cat_dummies], axis=1)
df.drop(columns=['category_top20'], inplace=True)

print(f"Dummies created: {cat_dummies.shape[1]}")
print(f"Names: {list(cat_dummies.columns)}")




   ---







   ## CONVERSION `weekends_only` and `workdays_only` → 0/1

In [ ]:
print("Unique values weekends_only:", df['weekends_only'].unique())
print("Type:", df['weekends_only'].dtype)
print()
print("Unique values workdays_only:", df['workdays_only'].unique())
print("Type:", df['workdays_only'].dtype)



In [ ]:
# String → numeric conversion
df['weekends_only'] = df['weekends_only'].astype(str).str.strip().replace({'True': 1, 'False': 0, 'nan': np.nan}).astype(float)
df['workdays_only'] = df['workdays_only'].astype(str).str.strip().replace({'True': 1, 'False': 0, 'nan': np.nan}).astype(float)

# NaNs remain NaN — will be imputed with median in Pipeline
# (which will be ~0 since True is very rare: 34 and 36 observations)
print("weekends_only after conversion:")
print(df['weekends_only'].value_counts(dropna=False))
print("\nworkdays_only after conversion:")
print(df['workdays_only'].value_counts(dropna=False))

# Add these two to the Group 2 list (median in Pipeline)
group2_median_pipeline.extend(['weekends_only', 'workdays_only'])




   ---







   ## FINAL CHECK

In [ ]:
# Separate train and test again
train_clean = df[df['_is_train'] == 1].copy()
test_clean  = df[df['_is_train'] == 0].copy()

# Remove auxiliary columns
train_clean.drop(columns=['_is_train', 'status_closed'], inplace=True)
test_clean.drop(columns=['_is_train', 'status_closed'], inplace=True)

# Re-add target to train
train_clean['status_closed'] = target.values

print(f"Clean train: {train_clean.shape}")
print(f"Clean test:  {test_clean.shape}")




In [ ]:
# Check how many NaNs are left (will need to be handled by the Pipeline)
remaining_na = train_clean.drop(columns=['restaurant_id', 'status_closed']).isna().sum()
remaining_na = remaining_na[remaining_na > 0].sort_values(ascending=False)

print(f"\nColumns with residual NaNs (to be handled in Pipeline): {len(remaining_na)}")
print(remaining_na)




In [ ]:
# Check types — everything must be numeric (int/float/bool)
non_numeric = train_clean.select_dtypes(exclude=[np.number, bool]).columns.tolist()
non_numeric = [c for c in non_numeric if c != 'restaurant_id']
if non_numeric:
    print(f"\n⚠️ WARNING: non-numeric columns remaining: {non_numeric}")
else:
    print("\n✅ All columns (except restaurant_id) are numeric.")




In [ ]:
# Column summary
print(f"\nTotal number of features (excluding id and target): "
      f"{train_clean.shape[1] - 2}")  # -2 for restaurant_id and status_closed
print(f"\nColumn list:")
feature_cols = [c for c in train_clean.columns if c not in ['restaurant_id', 'status_closed']]
for i, c in enumerate(feature_cols):
    print(f"  {i+1:3d}. {c}")




   ---







   ## SAVING







   Save the clean datasets for the next step.

In [ ]:
train_clean.to_csv('Data/restaurants_train_clean.csv', index=False)
test_clean.to_csv('Data/restaurants_test_clean.csv', index=False)
print("✅ Files saved: restaurants_train_clean.csv, restaurants_test_clean.csv")




   ---







   ## APPLIED STRATEGY SUMMARY















   | Group | Variables | Strategy | Where |







   |-------|-----------|-----------|------|







   | 0 | `is_new_restaurant` | Created from `user_ratings_total == 0` | Pre-Pipeline |







   | 1 | Ratings, review counts, linguistics (counts) | Fillna(0) | Pre-Pipeline |







   | 2 | Hours, temporal average ratings, linguistic average ratings, spatial (rating/age), place_age_days | Median | In Pipeline |







   | 3 | `price_level` | OHE with "missing" dummy | Pre-Pipeline |







   | 4 | `tagcat_*` | Fillna(0) | Pre-Pipeline |







   | 5 | `catch_restaurant_count_*` | Fillna(0) | Pre-Pipeline |







   | 6 | `category_top20` | OHE with "missing" dummy | Pre-Pipeline |







   | — | `weekends_only`, `workdays_only` | str→0/1 conversion, NaN→median | Pre + Pipeline |

  # === 03_feature_engineering.py ===



  Feature Engineering — 8 agreed new features



  Input: restaurants_train_clean.csv, restaurants_test_clean.csv



  Output: restaurants_train_fe.csv, restaurants_test_fe.csv

In [ ]:
train = pd.read_csv('Data/restaurants_train_clean.csv')
test  = pd.read_csv('Data/restaurants_test_clean.csv')

target = train['status_closed'].copy()
train.drop(columns=['status_closed'], inplace=True)

df = pd.concat([train, test], axis=0, ignore_index=True)
print(f"Combined dataset: {df.shape}")



In [ ]:
# =====================================================================
# 1. POI MARGINAL RINGS
# From concentric circles to bands: how many POIs are *between* two radii
# =====================================================================
df['poi_ring_100_200']   = df['poi_count_200m']  - df['poi_count_100m']
df['poi_ring_200_500']   = df['poi_count_500m']  - df['poi_count_200m']
df['poi_ring_500_1000']  = df['poi_count_1000m'] - df['poi_count_500m']
df['poi_ring_1000_2000'] = df['poi_count_2000m'] - df['poi_count_1000m']

print("1. POI rings created: poi_ring_100_200, poi_ring_200_500, poi_ring_500_1000, poi_ring_1000_2000")



In [ ]:
# =====================================================================
# 2. CATCH_RESTAURANT_COUNT MARGINAL RINGS
# Same logic for competing restaurants nearby
# =====================================================================
df['catch_ring_500_1000'] = df['catch_restaurant_count_1000m'] - df['catch_restaurant_count_500m']
df['catch_ring_1000_2000'] = df['catch_restaurant_count_2000m'] - df['catch_restaurant_count_1000m']

print("2. Catch rings created: catch_ring_500_1000, catch_ring_1000_2000")



In [ ]:
# =====================================================================
# 3. PLACE_AGE_DAYS² (quadratic term)
# Captures non-linear relationship: closure more likely for very young
# and very old places (U-curve)
# =====================================================================
df['place_age_days_sq'] = df['place_age_days'] ** 2

print("3. place_age_days_sq created")



In [ ]:
# =====================================================================
# 4. BAYESIAN RATING
# Rating adjusted for the number of reviews
# Formula: (rating_avg * n + global_mean * m) / (n + m)
# m = 10 (prior weight)
# =====================================================================
m = 10
global_mean = df['rating_avg'].mean()
df['bayesian_rating'] = (
    (df['rating_avg'] * df['user_ratings_total'] + global_mean * m) /
    (df['user_ratings_total'] + m)
)

print(f"4. bayesian_rating created (global_mean={global_mean:.4f}, m={m})")



In [ ]:
# =====================================================================
# 5. REVIEW TREND
# Difference between recent rating (3 months) and overall historical rating
# Negative = declining restaurant → closure more likely
# =====================================================================
df['review_trend'] = df['ratings_avg_3m_prior'] - df['rating_avg']

print("5. review_trend created (ratings_avg_3m_prior - rating_avg)")



In [ ]:
# =====================================================================
# 6. RATING CV (coefficient of variation)
# Normalized dispersion: std / mean
# Division by zero handling: if rating_avg == 0 → CV = 0
# =====================================================================
df['rating_cv'] = np.where(
    df['rating_avg'] > 0,
    df['rating_std'] / df['rating_avg'],
    0
)

print("6. rating_cv created (rating_std / rating_avg)")



In [ ]:
# =====================================================================
# 7. POLARIZATION
# Share of extreme reviews (1 and 5 stars) out of the total
# Highly polarizing place → potential fragility
# Division by zero handling: if user_ratings_total == 0 → 0
# =====================================================================
df['polarization'] = np.where(
    df['user_ratings_total'] > 0,
    (df['rating_5'] + df['rating_1']) / df['user_ratings_total'],
    0
)

print("7. polarization created ((rating_5 + rating_1) / user_ratings_total)")



In [ ]:
# =====================================================================
# 8. WEEKEND DEPENDENCY
# How much the place depends on the weekend
# Division by zero handling: if hours_open == 0 or NaN → NaN
# (will be imputed with median in the Pipeline)
# =====================================================================
df['weekend_dependency'] = np.where(
    (df['hours_open'] > 0) & (df['hours_open'].notna()),
    df['hours_open_weekends'] / df['hours_open'],
    np.nan
)

print("8. weekend_dependency created (hours_open_weekends / hours_open)")



In [ ]:
# =====================================================================
# VERIFICATION AND SAVING
# =====================================================================
print(f"\n{'='*60}")
print(f"Total features after engineering: {df.shape[1] - 2}")  # -2 for restaurant_id and _is_train
print(f"New features created: 8")

# Check the new features
new_features = [
    'poi_ring_100_200', 'poi_ring_200_500', 'poi_ring_500_1000', 'poi_ring_1000_2000',
    'catch_ring_500_1000', 'catch_ring_1000_2000',
    'place_age_days_sq', 'bayesian_rating', 'review_trend',
    'rating_cv', 'polarization', 'weekend_dependency'
]

print(f"\nNew features statistics:")
print(df[new_features].describe().round(4).to_string())

print(f"\nNaNs in new features:")
print(df[new_features].isna().sum().to_string())



In [ ]:
# Separate train and test
n_train = len(train)
train_fe = df.iloc[:n_train].copy()
test_fe  = df.iloc[n_train:].copy()

# Re-add target
train_fe['status_closed'] = target.values

print(f"\nFinal train: {train_fe.shape}")
print(f"Final test:  {test_fe.shape}")

# Saving
train_fe.to_csv('Data/restaurants_train_fe.csv', index=False)
test_fe.to_csv('Data/restaurants_test_fe.csv', index=False)
print("\n✅ Files saved: Data/restaurants_train_fe.csv, Data/restaurants_test_fe.csv")



  # === 03b — Advanced Feature Engineering ===







  **Machine Learning 1 — Prof. Piotr Wójcik — a.y. 2025/2026**







   This script applies advanced feature engineering techniques:



  1. Information Value (IV) — ranking of all features by predictive power



  2. WoE (Weight of Evidence) — supervised transformation of top variables



  3. WoE on category_top20 — replacement of 21 dummies with a single WoE



  4. Interactions between Strong variables







  Input:  restaurants_train_fe.csv, restaurants_test_fe.csv



  Output: restaurants_train_fe2.csv, restaurants_test_fe2.csv

In [ ]:
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 130)
pd.set_option('display.float_format', '{:.4f}'.format)



In [ ]:
# === LOADING ===
train = pd.read_csv('Data/restaurants_train_fe.csv')
test  = pd.read_csv('Data/restaurants_test_fe.csv')

target = train['status_closed'].values

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
print(f"Target: 0 → {(target==0).sum()}, 1 → {(target==1).sum()} ({target.mean()*100:.1f}%)")



  # ---



  # ## PART 1 — Information Value (IV) of all features







  IV measures the predictive power of each variable with respect to the target.



  Reference scale:



  - IV < 0.02: Not useful



  - 0.02 – 0.1: Weak



  - 0.1 – 0.3: Medium



  - 0.3 – 0.5: Strong



  - > 0.5: Suspicious (possible data leakage)

In [ ]:
from optbinning import OptimalBinning
features = train.drop(columns=['restaurant_id', 'status_closed'])

iv_results = []

for col in features.columns:
    x = features[col].values
    n_unique = pd.Series(x).dropna().nunique()
    
    dtype = "categorical" if n_unique <= 2 else "numerical"
    
    try:
        optb = OptimalBinning(
            name=col, dtype=dtype, solver="cp", max_n_prebins=20
        )
        optb.fit(x, target)
        
        binning_table = optb.binning_table.build()
        iv_total = binning_table["IV"].iloc[-1]
        n_bins = len(binning_table) - 1
        
        iv_results.append({
            'variable': col, 'IV': iv_total,
            'n_bins': n_bins, 'dtype': dtype
        })
    except:
        iv_results.append({
            'variable': col, 'IV': np.nan,
            'n_bins': np.nan, 'dtype': dtype
        })

iv_df = pd.DataFrame(iv_results).sort_values('IV', ascending=False).reset_index(drop=True)

def iv_category(iv):
    if pd.isna(iv): return "N/A"
    if iv < 0.02: return "Not useful"
    if iv < 0.1: return "Weak"
    if iv < 0.3: return "Medium"
    if iv < 0.5: return "Strong"
    return "Suspicious"

iv_df['predictive_power'] = iv_df['IV'].apply(iv_category)

print("=" * 80)
print("INFORMATION VALUE RANKING — Top 40")
print("=" * 80)
print(iv_df.head(40).to_string(index=False))

print("\n" + "=" * 80)
print("SUMMARY BY CATEGORY")
print("=" * 80)
print(iv_df['predictive_power'].value_counts().to_string())

print("\n" + "=" * 80)
print("NOT USEFUL VARIABLES (IV < 0.02)")
print("=" * 80)
print(iv_df[iv_df['IV'] < 0.02][['variable', 'IV']].to_string(index=False))

iv_df.to_csv('Data/03b_iv_results.csv', index=False)
print("\n✅ IV results saved to Data/03b_iv_results.csv")



  # ---



  # ## PART 2 — WoE on Top variables (Strong/Medium with non-linear relationship)







  Apply WoE Optimal Binning to variables with higher IV and many bins,



  where the relationship with the target is non-linear and WoE can capture it better.







  Strategy: create the WoE version NEXT TO the original.



  LASSO will then decide whether to keep or discard it.







  Variables selected for WoE:



  - user_ratings_total (IV=0.39, 14 bins)



  - place_age_days (IV=0.25, 12 bins)



  - tagcat_amenities (IV=0.27, 10 bins)



  - review_length_avg (IV=0.09, 13 bins — Weak but with many bins)



  - polarization (IV=0.30, 10 bins)



  - review_has_text_pct (IV=0.10, 11 bins)

In [ ]:
woe_candidates = [
    'user_ratings_total',
    'place_age_days',
    'tagcat_amenities',
    'review_length_avg',
    'polarization',
    'review_has_text_pct',
]

# Dictionary to save OptimalBinning models (needed to transform the test set)
woe_models = {}

for col in woe_candidates:
    x_train = train[col].values
    
    optb = OptimalBinning(
        name=col, dtype="numerical", solver="cp", max_n_prebins=20
    )
    optb.fit(x_train, target)
    
    # Save the model to reuse on the test set
    woe_models[col] = optb
    
    # Transform train and test
    woe_col_name = f"{col}_woe"
    train[woe_col_name] = optb.transform(x_train, metric="woe")
    test[woe_col_name]  = optb.transform(test[col].values, metric="woe")
    
    # Print binning table for analysis
    bt = optb.binning_table.build()
    print(f"\n{'='*60}")
    print(f"WoE Binning Table: {col}")
    print(f"{'='*60}")
    print(bt[['Bin', 'Count', 'Count (%)', 'Event rate', 'WoE', 'IV']].to_string())
    print(f"Split points: {optb.splits}")

print(f"\n✅ Created {len(woe_candidates)} new WoE features")



In [ ]:
# === PART 2b — WoE on additional Strong and Medium variables ===
woe_candidates_new = [
    # Strong (IV > 0.3) — not yet WoE
    'ratings_num_9m_prior',
    'ratings_num_12m_prior',
    'ratings_num_3m_prior',
    'ratings_num_6m_prior',
    'ratings_num_1m_prior',
    'rating_pl',
    'lang_pl_count',
    'ratings_avg_9m_prior',
    'ratings_avg_6m_prior',
    'rating_foreign',
    'ratings_avg_1m_prior',
    'ratings_avg_3m_prior',
    # Medium (IV 0.1–0.3) — not yet WoE
    'review_trend',
    'ratings_avg_12m_prior',
    'first_review_year_max',
    'rating_mean_foreign',
    'tagcat_atmosphere',
    'rating_mean_lang_ratio',
    'rating_mean_pl',
    'rating_4',
    'rating_3',
    'tagcat_offerings',
    'rating_std',
    'foreign_lang_share',
    'review_length_std',
]

for col in woe_candidates_new:
    x_train = train[col].values
    
    try:
        optb = OptimalBinning(
            name=col, dtype="numerical", solver="cp", max_n_prebins=20
        )
        optb.fit(x_train, target)
        
        woe_col_name = f"{col}_woe"
        train[woe_col_name] = optb.transform(x_train, metric="woe")
        test[woe_col_name]  = optb.transform(test[col].values, metric="woe")
        
        woe_models[col] = optb
        
        bt = optb.binning_table.build()
        print(f"\n{'='*60}")
        print(f"WoE Binning Table: {col}")
        print(f"{'='*60}")
        print(bt[['Bin', 'Count', 'Count (%)', 'Event rate', 'WoE', 'IV']].to_string())
        print(f"Split points: {optb.splits}")
    except Exception as e:
        print(f"\n⚠️ ERROR on {col}: {e}")

print(f"\n✅ Created {len(woe_candidates_new)} new WoE features (batch 2)")
print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")


  # ---



  # ## PART 3 — WoE on category_top20 (replacement of 21 dummies)







  The category_top20 dummies have almost all IV = 0.



  WoE on categorical variable can capture the signal



  by aggregating similar categories into a few bins.







  Strategy: reconstruct category_top20 from dummies,



  apply categorical WoE, and drop all 21 dummies.

In [ ]:
# Reconstruct category_top20 from dummies
cat_dummies = [c for c in train.columns if c.startswith('cat_')]
print(f"Category dummies found: {len(cat_dummies)}")
print(cat_dummies)

# For each row, find which dummy is = 1
def reconstruct_category(row):
    for col in cat_dummies:
        if row[col] == 1:
            return col.replace('cat_', '')
    return 'missing'

train['_category_top20'] = train[cat_dummies].apply(reconstruct_category, axis=1)
test['_category_top20']  = test[cat_dummies].apply(reconstruct_category, axis=1)

print(f"\nReconstructed categories (train):")
print(train['_category_top20'].value_counts().to_string())



In [ ]:
# Apply categorical WoE
optb_cat = OptimalBinning(
    name="category_top20",
    dtype="categorical",
    solver="cp"
)

optb_cat.fit(train['_category_top20'].values, target)

# Binning table
bt_cat = optb_cat.binning_table.build()
print(f"\n{'='*60}")
print(f"WoE Binning Table: category_top20 (categorical)")
print(f"{'='*60}")
print(bt_cat[['Bin', 'Count', 'Count (%)', 'Event rate', 'WoE', 'IV']].to_string())

# Transform
train['category_woe'] = optb_cat.transform(train['_category_top20'].values, metric="woe")
test['category_woe']  = optb_cat.transform(test['_category_top20'].values, metric="woe")

# Drop the 21 dummies and the temporary column
train.drop(columns=cat_dummies + ['_category_top20'], inplace=True)
test.drop(columns=cat_dummies + ['_category_top20'], inplace=True)

print(f"\n✅ Dropped {len(cat_dummies)} category dummies, created 'category_woe'")



  # ---



  # ## PART 4 — Interactions between Strong variables







  Create interactions (products and ratios) between



  variables with higher predictive power. These can capture



  combined effects that single variables miss.

In [ ]:
# --- Interaction 1: Volume × Quality ---
# Restaurants with many reviews AND low rating → higher risk
train['vol_x_rating'] = train['user_ratings_total'] * train['rating_avg']
test['vol_x_rating']  = test['user_ratings_total'] * test['rating_avg']

# --- Interaction 2: Recent trend × Volume ---
# A negative trend weighs more if the restaurant has many reviews
train['trend_x_vol'] = train['review_trend'] * train['user_ratings_total']
test['trend_x_vol']  = test['review_trend'] * test['user_ratings_total']

# --- Interaction 3: Polarization × Amenities ---
# Polarizing restaurants with few amenities → high risk
train['polar_x_amenities'] = train['polarization'] * train['tagcat_amenities']
test['polar_x_amenities']  = test['polarization'] * test['tagcat_amenities']

# --- Interaction 4: PL vs Foreign Rating (difference) ---
# Large difference between local and foreign ratings → signal
train['rating_gap_pl_foreign'] = train['rating_mean_pl'] - train['rating_mean_foreign']
test['rating_gap_pl_foreign']  = test['rating_mean_pl'] - test['rating_mean_foreign']

# --- Interaction 5: Age × Average Rating ---
# Old restaurant with low rating → high risk
train['age_x_rating'] = train['place_age_days'] * train['rating_avg']
test['age_x_rating']  = test['place_age_days'] * test['rating_avg']

# --- Interaction 6: Weighted competitive density ---
# Many nearby competitors with high rating → more competition
train['competition_pressure'] = train['catch_restaurant_count_500m'] * train['catch_rating_avg_500m']
test['competition_pressure']  = test['catch_restaurant_count_500m'] * test['catch_rating_avg_500m']

print("✅ Created 6 interaction features:")
interaction_cols = ['vol_x_rating', 'trend_x_vol', 'polar_x_amenities',
                    'rating_gap_pl_foreign', 'age_x_rating', 'competition_pressure']
for col in interaction_cols:
    print(f"  - {col}: train mean={train[col].mean():.4f}, std={train[col].std():.4f}")



  # ---



  # ## SUMMARY AND SAVING





In [ ]:
print(f"\n{'='*60}")
print(f"ADVANCED FEATURE ENGINEERING SUMMARY")
print(f"{'='*60}")
print(f"Original dataset: train {pd.read_csv('Data/restaurants_train_fe.csv').shape[1]} columns")
print(f"Dataset after FE2:  train {train.shape[1]} columns")
print(f"\nNew features created:")
print(f"  - {len(woe_candidates)} WoE features (top numeric variables)")
print(f"  - 1 categorical WoE feature (category_woe)")
print(f"  - 6 interaction features")
print(f"  - Dropped: {len(cat_dummies)} category dummies (replaced by category_woe)")
print(f"\nFinal train columns: {train.shape[1]}")
print(f"Final test columns:  {test.shape[1]}")

# Check column consistency
train_cols = set(train.columns) - {'status_closed'}
test_cols  = set(test.columns)
assert train_cols == test_cols, f"Mismatch! Only in train: {train_cols - test_cols}, Only in test: {test_cols - train_cols}"
print("\n✅ Train and test columns are consistent")



In [ ]:
# === SAVING ===
train.to_csv('Data/restaurants_train_fe2.csv', index=False)
test.to_csv('Data/restaurants_test_fe2.csv', index=False)

print(f"\n✅ Saved:")
print(f"  - Data/restaurants_train_fe2.csv ({train.shape})")
print(f"  - Data/restaurants_test_fe2.csv ({test.shape})")



In [ ]:
check = pd.read_csv('Data/restaurants_train_fe2.csv')
print("weekends_only in fe2 file:")
print(f"  dtype: {check['weekends_only'].dtype}")
print(f"  unique: {check['weekends_only'].unique()[:10]}")
print(f"  value_counts:")
print(check['weekends_only'].value_counts(dropna=False))



In [ ]:
# Check 1: in clean file (step 2 output)
clean = pd.read_csv('Data/restaurants_train_clean.csv')
print("=== CLEAN (step 2 output) ===")
print(clean['weekends_only'].value_counts(dropna=False))

# Check 2: in fe file (step 3 output)
fe = pd.read_csv('Data/restaurants_train_fe.csv')
print("\n=== FE (step 3 output) ===")
print(fe['weekends_only'].value_counts(dropna=False))



  # # === 03c — Revision of Imputation Strategy ===







  **Machine Learning 1 — Prof. Piotr Wójcik — a.y. 2025/2026**







  Compare 3 imputation strategies for Group 2 variables



  (those left with NaNs, currently imputed with median in the Pipeline):







  - Pipeline A: SimpleImputer(median) — current baseline



  - Pipeline B: KNNImputer(n_neighbors=5)



  - Pipeline C: SimpleImputer(median) + Missing Flags







  All use the same ElasticNet model and the same CV.







  Input: restaurants_train_fe2.csv



  Output: comparative results + possible dataset update

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)



In [ ]:
train = pd.read_csv('Data/restaurants_train_fe2.csv')

target = train['status_closed']
X = train.drop(columns=['restaurant_id', 'status_closed'])

print(f"Train: {X.shape}")
print(f"Target: {target.value_counts().to_dict()}")

# === IDENTIFICATION OF VARIABLES WITH MISSINGS ===
missing_counts = X.isnull().sum()
missing_cols = missing_counts[missing_counts > 0].sort_values(ascending=False)

print("Variables with NaNs in the dataset:")
for col, count in missing_cols.items():
    pct = count / len(X) * 100
    print(f"  {col}: {count} NaNs ({pct:.1f}%)")
print(f"\nTotal variables with NaNs: {len(missing_cols)}")



In [ ]:
# === CV SETUP (identical to the one used so far) ===
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Model identical to our current winner
model_params = dict(
    penalty='elasticnet', solver='saga', C=0.008,
    l1_ratio=0.9, class_weight='balanced',
    max_iter=5000, random_state=42
)



  # ---



  # ## Pipeline A — SimpleImputer(median) — Current baseline

In [ ]:
pipeline_A = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', PowerTransformer(method='yeo-johnson')),
    ('model', LogisticRegression(**model_params))
])

scores_A = cross_val_score(pipeline_A, X, target, cv=cv,
                           scoring='balanced_accuracy', n_jobs=-1)

print(f"Pipeline A — Median")
print(f"  Fold scores: {scores_A}")
print(f"  Mean BA: {scores_A.mean():.4f} (±{scores_A.std():.4f})")



  # ---



  # ## Pipeline B — KNNImputer







  The KNN Imputer finds the K most similar neighbors (based on



  non-missing features) and imputes with the mean of their values.



  Note: it is slower than the median, but can capture local patterns.

In [ ]:
pipeline_B = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5, weights='distance')),
    ('scaler', PowerTransformer(method='yeo-johnson')),
    ('model', LogisticRegression(**model_params))
])

scores_B = cross_val_score(pipeline_B, X, target, cv=cv,
                           scoring='balanced_accuracy', n_jobs=-1)

print(f"Pipeline B — KNN Imputer (k=5, distance)")
print(f"  Fold scores: {scores_B}")
print(f"  Mean BA: {scores_B.mean():.4f} (±{scores_B.std():.4f})")



  # ---



  # ## Pipeline C — Median + Missing Flags







  Create binary columns indicating where a NaN was.



  So the model can learn that "data was missing" is a signal



  (as we saw: place_age_days missing → 29% event rate!).



  Imputation remains median, but the flag preserves the info.

In [ ]:
# Create missing flags ONLY for variables with enough NaNs
# (at least 1% of dataset, otherwise the flag is too rare to be useful)
min_missing_pct = 0.01
flag_candidates = [col for col, count in missing_cols.items()
                   if count / len(X) >= min_missing_pct]

print(f"Candidate variables for missing flag (>= {min_missing_pct*100:.0f}% NaNs):")
for col in flag_candidates:
    pct = X[col].isnull().mean() * 100
    print(f"  {col}: {pct:.1f}%")

# Create the version with flags
X_with_flags = X.copy()
for col in flag_candidates:
    X_with_flags[f'{col}_missing'] = X_with_flags[col].isnull().astype(int)

print(f"\nAdded columns: {len(flag_candidates)} missing flags")
print(f"Original X shape: {X.shape}")
print(f"X with flags shape: {X_with_flags.shape}")



In [ ]:
pipeline_C = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', PowerTransformer(method='yeo-johnson')),
    ('model', LogisticRegression(**model_params))
])

scores_C = cross_val_score(pipeline_C, X_with_flags, target, cv=cv,
                           scoring='balanced_accuracy', n_jobs=-1)

print(f"Pipeline C — Median + Missing Flags")
print(f"  Fold scores: {scores_C}")
print(f"  Mean BA: {scores_C.mean():.4f} (±{scores_C.std():.4f})")



  # ---



  # ## COMPARISON OF IMPUTATION STRATEGIES

In [ ]:
print("=" * 60)
print("COMPARISON OF IMPUTATION STRATEGIES")
print("=" * 60)

results = pd.DataFrame({
    'Pipeline': ['A — Median (baseline)', 'B — KNN Imputer', 'C — Median + Flags'],
    'Mean BA': [scores_A.mean(), scores_B.mean(), scores_C.mean()],
    'Std BA': [scores_A.std(), scores_B.std(), scores_C.std()],
    'Fold 1': [scores_A[0], scores_B[0], scores_C[0]],
    'Fold 2': [scores_A[1], scores_B[1], scores_C[1]],
    'Fold 3': [scores_A[2], scores_B[2], scores_C[2]],
    'Fold 4': [scores_A[3], scores_B[3], scores_C[3]],
    'Fold 5': [scores_A[4], scores_B[4], scores_C[4]],
})

results = results.sort_values('Mean BA', ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

best = results.iloc[0]['Pipeline']
best_score = results.iloc[0]['Mean BA']
baseline_score = scores_A.mean()
diff = best_score - baseline_score

print(f"\n→ Best: {best} (BA = {best_score:.4f})")
if diff > 0:
    print(f"  Improvement over baseline: +{diff*100:.2f} pp")
else:
    print(f"  The baseline is already the best or equivalent")



  # === 04_correlation_pruning.py ===







  Correlation pruning — removal of variables with ρ > 0.90



  Input: restaurants_train_fe.csv, restaurants_test_fe.csv



  Output: restaurants_train_pruned.csv, restaurants_test_pruned.csv

In [ ]:
# === LOADING ===
train = pd.read_csv('Data/restaurants_train_fe2.csv')
test  = pd.read_csv('Data/restaurants_test_fe2.csv')

target = train['status_closed']
features = train.drop(columns=['restaurant_id', 'status_closed']).columns.tolist()

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
print(f"Feature: {len(features)}")



In [ ]:
# === IV LOADING ===
iv_df = pd.read_csv('Data/03b_iv_results.csv')
iv_dict = dict(zip(iv_df['variable'], iv_df['IV']))

# === DEFINITION OF PROTECTED FEATURES ===
# 1) Variables with IV >= 0.10 → NEVER touched
protected_by_iv = [v for v, iv in iv_dict.items() if iv >= 0.10]

# 2) New features (WoE, interactions) → NOT touched
new_features = [c for c in features if c.endswith('_woe') or
                c in ['vol_x_rating', 'trend_x_vol', 'polar_x_amenities',
                       'rating_gap_pl_foreign', 'age_x_rating',
                       'competition_pressure', 'category_woe']]

protected = set(protected_by_iv + new_features)

print(f"Protected features (IV >= 0.10): {len(protected_by_iv)}")
print(f"New features (WoE/interactions): {len(new_features)}")
print(f"Total protected: {len(protected)}")




In [ ]:
# === CORRELATION MATRIX ===
X = train[features]
X_temp = X.fillna(X.median())
corr_matrix = X_temp.corr().abs()

# Pairs with |ρ| > 0.90
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > 0.90:
            high_corr_pairs.append({
                'var_1': corr_matrix.columns[i],
                'var_2': corr_matrix.columns[j],
                'correlation': corr_matrix.iloc[i, j]
            })

pairs_df = pd.DataFrame(high_corr_pairs).sort_values('correlation', ascending=False)
print(f"\nPairs with |p| > 0.90: {len(pairs_df)}")




In [ ]:
# === PRUNING DECISIONS WITH PROTECTION ===
print("\nPruning decisions:")
print("-" * 80)

to_drop = []
skipped = []

for _, row in pairs_df.iterrows():
    v1, v2, rho = row['var_1'], row['var_2'], row['correlation']
    v1_protected = v1 in protected
    v2_protected = v2 in protected

    # Case 1: both protected → none is touched
    if v1_protected and v2_protected:
        skipped.append((v1, v2, rho, "both protected"))
        continue

    # Case 2: one protected, one not → drop the unprotected one
    if v1_protected and not v2_protected:
        to_drop.append(v2)
        print(f"  ρ={rho:.3f}: KEEP {v1} (protected) | DROP {v2}")
        continue
    if v2_protected and not v1_protected:
        to_drop.append(v1)
        print(f"  ρ={rho:.3f}: KEEP {v2} (protected) | DROP {v1}")
        continue

    # Case 3: none protected → drop the one with lower IV
    iv1 = iv_dict.get(v1, 0)
    iv2 = iv_dict.get(v2, 0)
    drop = v1 if iv1 < iv2 else v2
    keep = v2 if iv1 < iv2 else v1
    to_drop.append(drop)
    print(f"  ρ={rho:.3f}: KEEP {keep} (IV={iv_dict.get(keep, 0):.4f}) | DROP {drop} (IV={iv_dict.get(drop, 0):.4f})")

# Remove duplicates
to_drop = list(set(to_drop))

print(f"\n--- SKIPPED pairs (both protected): {len(skipped)} ---")
for v1, v2, rho, reason in skipped:
    print(f"  ρ={rho:.3f}: {v1} ↔ {v2} → {reason}")

print(f"\n--- Variables to drop: {len(to_drop)} ---")
for col in sorted(to_drop):
    iv_val = iv_dict.get(col, 0)
    print(f"  - {col} (IV={iv_val:.4f})")




In [ ]:
# === PRUNING APPLICATION ===
train.drop(columns=to_drop, inplace=True)
test.drop(columns=to_drop, inplace=True)

features_pruned = [c for c in train.columns if c not in ['restaurant_id', 'status_closed']]

print(f"\nAfter correlation pruning:")
print(f"  Train: {train.shape}")
print(f"  Test:  {test.shape}")
print(f"  Remaining features: {len(features_pruned)}")




  # ---



  # # === 04b — Feature Selection with LASSO ===

In [ ]:
X_train = train.drop(columns=['restaurant_id', 'status_closed'])
y_train = target

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)



In [ ]:
# === STEP 1: Optimal C for LASSO (reduced range since we know it's ~0.13) ===
C_values = np.logspace(-3, 1, 20)

lasso_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', PowerTransformer(method='yeo-johnson')),
    ('model', LogisticRegression(
        penalty='l1', solver='saga',
        class_weight='balanced',
        max_iter=5000, random_state=42
    ))
])

print("Searching for optimal C for LASSO...")
print("-" * 50)

lasso_results = []
for C in C_values:
    lasso_pipeline.set_params(model__C=C)
    scores = cross_val_score(lasso_pipeline, X_train, y_train,
                             cv=cv, scoring='balanced_accuracy', n_jobs=-1)
    lasso_results.append({'C': C, 'mean_ba': scores.mean(), 'std_ba': scores.std()})
    print(f"  C={C:.6f}  →  BA={scores.mean():.4f} (±{scores.std():.4f})")

lasso_results_df = pd.DataFrame(lasso_results)
best_C = lasso_results_df.loc[lasso_results_df['mean_ba'].idxmax(), 'C']
best_BA = lasso_results_df['mean_ba'].max()
print(f"\nBest: C={best_C:.6f}, BA={best_BA:.4f}")




In [ ]:
best_C = best_C

# === STEP 2: Extract coefficients ===
lasso_pipeline.set_params(model__C=best_C)
lasso_pipeline.fit(X_train, y_train)

lasso_model = lasso_pipeline.named_steps['model']
coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': lasso_model.coef_[0],
    'abs_coef': np.abs(lasso_model.coef_[0])
}).sort_values('abs_coef', ascending=False)

n_kept = (coef_df['coefficient'] != 0).sum()
n_dropped = (coef_df['coefficient'] == 0).sum()

print(f"\n{'='*60}")
print(f"LASSO RESULTS (C={best_C:.6f})")
print(f"{'='*60}")
print(f"Kept features (coeff ≠ 0): {n_kept}")
print(f"Zeroed features (coeff = 0): {n_dropped}")

print(f"\n--- Top 20 features by importance ---")
print(coef_df.head(20).to_string(index=False))

print(f"\n--- Features zeroed by LASSO ---")
zeroed = coef_df[coef_df['coefficient'] == 0]['feature'].tolist()
print(f"Total: {len(zeroed)}")
for f in zeroed:
    print(f"  - {f}")




In [ ]:
# === STEP 3: Comparison ===
selected_features = coef_df[coef_df['coefficient'] != 0]['feature'].tolist()
X_train_selected = X_train[selected_features]

elasticnet_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', PowerTransformer(method='yeo-johnson')),
    ('model', LogisticRegression(
        penalty='elasticnet', solver='saga', C=0.008,
        l1_ratio=0.9, class_weight='balanced',
        max_iter=5000, random_state=42
    ))
])

scores_all = cross_val_score(elasticnet_pipeline, X_train, y_train,
                             cv=cv, scoring='balanced_accuracy', n_jobs=-1)
scores_selected = cross_val_score(elasticnet_pipeline, X_train_selected, y_train,
                                  cv=cv, scoring='balanced_accuracy', n_jobs=-1)

print(f"\n{'='*60}")
print(f"COMPARISON: All features vs LASSO Selection")
print(f"{'='*60}")
print(f"All ({X_train.shape[1]} feat):      BA = {scores_all.mean():.4f} (±{scores_all.std():.4f})")
print(f"LASSO ({X_train_selected.shape[1]} feat): BA = {scores_selected.mean():.4f} (±{scores_selected.std():.4f})")

diff = scores_selected.mean() - scores_all.mean()
if diff > 0:
    print(f"\n→ LASSO selection improves by +{diff*100:.2f} pp")
elif diff > -0.002:
    print(f"\n→ Equivalent results, but with fewer features (more interpretable)")
else:
    print(f"\n→ All features work better, keep everything")




  # ---



  # ## SAVING — Uncomment the appropriate option

In [ ]:
# --- OPTION A: If LASSO improves or is equivalent ---
train_final = train[['restaurant_id'] + selected_features + ['status_closed']]
test_final  = test[['restaurant_id'] + selected_features]
train_final.to_csv('Data/restaurants_train_final.csv', index=False)
test_final.to_csv('Data/restaurants_test_final.csv', index=False)
print(f"✅ Saved datasets with {len(selected_features)} selected features")



  # === 05a_baseline_logistic.py ===







  Baseline — Logistic Regression with default parameters



  Establishes the minimum benchmark to beat with tuned models



  Input: restaurants_train_pruned.csv



  Metric: balanced_accuracy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (confusion_matrix, balanced_accuracy_score,
                             classification_report, make_scorer)
import warnings
warnings.filterwarnings('ignore')



In [ ]:
# === LOADING ===
train = pd.read_csv('Data/restaurants_train_final.csv')

target = train['status_closed']
X = train.drop(columns=['restaurant_id', 'status_closed'])

feature_names = X.columns.tolist()
print(f"Feature: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"Target distribution:\n{target.value_counts(normalize=True).round(4)}")



In [ ]:
# === PIPELINE ===
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        class_weight='balanced',
        solver='lbfgs',
        max_iter=5000,
        random_state=42
    ))
])

# === CROSS-VALIDATION ===
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'balanced_accuracy': 'balanced_accuracy',
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
}

print("\n⏳ Cross-validation in progress...")
cv_results = cross_validate(
    pipeline, X, target,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    return_estimator=True
)

# === RESULTS ===
print(f"\n{'='*60}")
print(f"BASELINE — Logistic Regression (default)")
print(f"{'='*60}")



In [ ]:
for metric in scoring.keys():
    test_scores = cv_results[f'test_{metric}']
    train_scores = cv_results[f'train_{metric}']
    print(f"\n{metric}:")
    print(f"  Train: {train_scores.mean():.4f} (±{train_scores.std():.4f})")
    print(f"  Test:  {test_scores.mean():.4f} (±{test_scores.std():.4f})")

# === AGGREGATED CONFUSION MATRIX ===
# Collect OOF (out-of-fold) predictions
oof_preds = np.zeros(len(target))
oof_proba = np.zeros(len(target))

for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X, target)):
    estimator = cv_results['estimator'][fold_idx]
    oof_preds[val_idx] = estimator.predict(X.iloc[val_idx])
    oof_proba[val_idx] = estimator.predict_proba(X.iloc[val_idx])[:, 1]

cm = confusion_matrix(target, oof_preds)
ba = balanced_accuracy_score(target, oof_preds)

print(f"\n{'='*60}")
print(f"Confusion Matrix (Aggregated OOF):")
print(f"{'='*60}")
print(f"                 Predicted 0   Predicted 1")
print(f"  Actual 0 (open):   {cm[0,0]:>6d}        {cm[0,1]:>6d}")
print(f"  Actual 1 (closed): {cm[1,0]:>6d}        {cm[1,1]:>6d}")
print(f"\nBalanced Accuracy OOF: {ba:.4f}")
print(f"\nClassification Report (OOF):")
print(classification_report(target, oof_preds, target_names=['Open', 'Closed']))



In [ ]:
# === PLOT 1: Confusion Matrix ===
fig1, ax1 = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Open', 'Closed'],
            yticklabels=['Open', 'Closed'])
ax1.set_title('BASELINE — Confusion Matrix (OOF)')
ax1.set_ylabel('Actual')
ax1.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('Data/05a_baseline_cm.png', dpi=150, bbox_inches='tight')
plt.show()

# === PLOT 2: Balanced Accuracy per Fold ===
fig2, ax2 = plt.subplots(figsize=(6, 5))
fold_scores = cv_results['test_balanced_accuracy']
fold_labels = [f'Fold {i+1}' for i in range(5)]
colors = ['#3498db' if s >= fold_scores.mean() else '#e74c3c' for s in fold_scores]
ax2.bar(fold_labels, fold_scores, color=colors, edgecolor='black', alpha=0.8)
ax2.axhline(y=fold_scores.mean(), color='black', linestyle='--',
            label=f'Mean: {fold_scores.mean():.4f}')
ax2.set_title('BASELINE — Balanced Accuracy per Fold')
ax2.set_ylabel('Balanced Accuracy')
ax2.set_ylim(fold_scores.min() - 0.02, fold_scores.max() + 0.02)
ax2.legend()
plt.tight_layout()
plt.savefig('Data/05a_baseline_folds.png', dpi=150, bbox_inches='tight')
plt.show()

# === PLOT 3: Top 20 Coefficients ===
fig3, ax3 = plt.subplots(figsize=(8, 6))
coefs = np.array([est.named_steps['model'].coef_[0] for est in cv_results['estimator']])
coef_mean = coefs.mean(axis=0)
coef_std  = coefs.std(axis=0)
top_idx = np.argsort(np.abs(coef_mean))[-20:]
top_names = [feature_names[i] for i in top_idx]
top_coefs = coef_mean[top_idx]
top_stds  = coef_std[top_idx]
color_coefs = ['#e74c3c' if c > 0 else '#2ecc71' for c in top_coefs]
ax3.barh(top_names, top_coefs, xerr=top_stds, color=color_coefs,
         edgecolor='black', alpha=0.8)
ax3.set_title('BASELINE — Top 20 Coefficients (CV mean)')
ax3.set_xlabel('Coefficient')
ax3.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('Data/05a_baseline_coefs.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# === SAVING RESULTS FOR COMPARISON ===
results_baseline = {
    'model': 'Baseline Logistic',
    'balanced_accuracy_mean': cv_results['test_balanced_accuracy'].mean(),
    'balanced_accuracy_std': cv_results['test_balanced_accuracy'].std(),
    'accuracy_mean': cv_results['test_accuracy'].mean(),
    'precision_mean': cv_results['test_precision'].mean(),
    'recall_mean': cv_results['test_recall'].mean(),
    'f1_mean': cv_results['test_f1'].mean(),
}

pd.DataFrame([results_baseline]).to_csv('Data/results_baseline.csv', index=False)
print("✅ Results saved: Data/results_baseline.csv")



  # === 05b_logistic_L1.py ===







  Logistic Regression L1 (LASSO) with GridSearchCV



  Input: restaurants_train_pruned.csv



  Metric: balanced_accuracy

In [ ]:
from sklearn.model_selection import GridSearchCV

# === PIPELINE ===
pipeline_l1 = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        penalty='l1',
        solver='saga',
        class_weight='balanced',
        max_iter=5000,
        random_state=42
    ))
])

# === GRID SEARCH ===
param_grid = {
    'model__C': [0.001, 0.01, 0.1, 1, 10]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n⏳ GridSearchCV in progress...")
grid = GridSearchCV(
    pipeline_l1,
    param_grid,
    cv=cv,
    scoring='balanced_accuracy',
    refit=True,
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)
grid.fit(X, target)



In [ ]:
# === RESULTS ===
print(f"\n{'='*60}")
print(f"LOGISTIC REGRESSION L1 (LASSO)")
print(f"{'='*60}")
print(f"Best C: {grid.best_params_['model__C']}")
print(f"Best Balanced Accuracy (CV): {grid.best_score_:.4f}")



In [ ]:
# Detail for each C
results_df = pd.DataFrame(grid.cv_results_)
for _, row in results_df.iterrows():
    c_val = row['param_model__C']
    print(f"  C={c_val:<6} → Train: {row['mean_train_score']:.4f}  Test: {row['mean_test_score']:.4f} (±{row['std_test_score']:.4f})")
    
best_model = grid.best_estimator_.named_steps['model']
print(best_model)    



In [ ]:
# === PLOT 1: Confusion Matrix ===
fig1, ax1 = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Open', 'Closed'],
            yticklabels=['Open', 'Closed'])
ax1.set_title(f'L1 LASSO (C={best_C}) — Confusion Matrix (OOF)')
ax1.set_ylabel('Actual')
ax1.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('Data/05b_L1_cm.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# === PLOT 2: Balanced Accuracy vs C ===
fig2, ax2 = plt.subplots(figsize=(8, 5))
c_values = results_df['param_model__C'].astype(float)
ax2.plot(c_values, results_df['mean_train_score'], 'o-', label='Train', color='#3498db')
ax2.plot(c_values, results_df['mean_test_score'], 'o-', label='Test', color='#e74c3c')
ax2.fill_between(c_values,
                 results_df['mean_test_score'] - results_df['std_test_score'],
                 results_df['mean_test_score'] + results_df['std_test_score'],
                 alpha=0.2, color='#e74c3c')
ax2.set_xscale('log')
ax2.set_xlabel('C (regularization)')
ax2.set_ylabel('Balanced Accuracy')
ax2.set_title('L1 LASSO — Balanced Accuracy vs C')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Data/05b_L1_tuning.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# === PLOT 3: Top 20 Coefficients ===
fig3, ax3 = plt.subplots(figsize=(8, 6))
coef_vals = best_model.coef_[0]
top_idx = np.argsort(np.abs(coef_vals))[-20:]
top_names = [feature_names[i] for i in top_idx]
top_coefs = coef_vals[top_idx]
color_coefs = ['#e74c3c' if c > 0 else '#2ecc71' for c in top_coefs]
ax3.barh(top_names, top_coefs, color=color_coefs, edgecolor='black', alpha=0.8)
ax3.set_title(f'L1 LASSO (C={best_C}) — Top 20 Coefficients')
ax3.set_xlabel('Coefficient')
ax3.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('Data/05b_L1_coefs.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# === SAVING RESULTS ===
results = {
    'model': 'Logistic L1 (LASSO)',
    'best_params': str(grid.best_params_),
    'balanced_accuracy_mean': grid.best_score_,
    'balanced_accuracy_std': results_df.loc[grid.best_index_, 'std_test_score'],
    'accuracy_mean': ba,
}
pd.DataFrame([results]).to_csv('Data/results_L1.csv', index=False)
print("\n✅ Results saved: Data/results_L1.csv")




  # === 05c_logistic_L2.py ===







  Logistic Regression L2 (Ridge) with GridSearchCV



  Input: restaurants_train_pruned.csv



  Metric: balanced_accuracy

In [ ]:
# === PIPELINE ===
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        penalty='l2',
        solver='lbfgs',
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    ))
])

# === GRID SEARCH ===
param_grid = {
    'model__C': [0.001, 0.01, 0.1, 1, 10]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n⏳ GridSearchCV in progress...")
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='balanced_accuracy',
    refit=True,
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)
grid.fit(X, target)




In [ ]:
# === RESULTS ===
print(f"\n{'='*60}")
print(f"LOGISTIC REGRESSION L2 (RIDGE)")
print(f"{'='*60}")
print(f"Best C: {grid.best_params_['model__C']}")
print(f"Best Balanced Accuracy (CV): {grid.best_score_:.4f}")

results_df = pd.DataFrame(grid.cv_results_)
for _, row in results_df.iterrows():
    c_val = row['param_model__C']
    print(f"  C={c_val:<6} → Train: {row['mean_train_score']:.4f}  Test: {row['mean_test_score']:.4f} (±{row['std_test_score']:.4f})")



In [ ]:
# === OOF PREDICTIONS ===
oof_preds = np.zeros(len(target))
oof_proba = np.zeros(len(target))

best_C = grid.best_params_['model__C']
for train_idx, val_idx in cv.split(X, target):
    pipe_fold = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            penalty='l2', solver='lbfgs', C=best_C,
            class_weight='balanced', max_iter=2000, random_state=42
        ))
    ])
    pipe_fold.fit(X.iloc[train_idx], target.iloc[train_idx])
    oof_preds[val_idx] = pipe_fold.predict(X.iloc[val_idx])
    oof_proba[val_idx] = pipe_fold.predict_proba(X.iloc[val_idx])[:, 1]

cm = confusion_matrix(target, oof_preds)
ba = balanced_accuracy_score(target, oof_preds)

print(f"\nConfusion Matrix (OOF):")
print(f"                 Predicted 0   Predicted 1")
print(f"  Actual 0 (open):   {cm[0,0]:>6d}        {cm[0,1]:>6d}")
print(f"  Actual 1 (closed): {cm[1,0]:>6d}        {cm[1,1]:>6d}")
print(f"\nBalanced Accuracy OOF: {ba:.4f}")
print(f"\nClassification Report (OOF):")
print(classification_report(target, oof_preds, target_names=['Open', 'Closed']))



In [ ]:
# === PLOT 1: Confusion Matrix ===
fig1, ax1 = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Open', 'Closed'],
            yticklabels=['Open', 'Closed'])
ax1.set_title(f'L2 Ridge (C={best_C}) — Confusion Matrix (OOF)')
ax1.set_ylabel('Actual')
ax1.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('Data/05c_L2_cm.png', dpi=150, bbox_inches='tight')
plt.show()

# === PLOT 2: Balanced Accuracy vs C ===
fig2, ax2 = plt.subplots(figsize=(8, 5))
c_values = results_df['param_model__C'].astype(float)
ax2.plot(c_values, results_df['mean_train_score'], 'o-', label='Train', color='#3498db')
ax2.plot(c_values, results_df['mean_test_score'], 'o-', label='Test', color='#e74c3c')
ax2.fill_between(c_values,
                 results_df['mean_test_score'] - results_df['std_test_score'],
                 results_df['mean_test_score'] + results_df['std_test_score'],
                 alpha=0.2, color='#e74c3c')
ax2.set_xscale('log')
ax2.set_xlabel('C (regularization)')
ax2.set_ylabel('Balanced Accuracy')
ax2.set_title('L2 Ridge — Balanced Accuracy vs C')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Data/05c_L2_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

# === PLOT 3: Top 20 Coefficients ===
fig3, ax3 = plt.subplots(figsize=(8, 6))
best_model = grid.best_estimator_.named_steps['model']
coef_vals = best_model.coef_[0]
top_idx = np.argsort(np.abs(coef_vals))[-20:]
top_names = [feature_names[i] for i in top_idx]
top_coefs = coef_vals[top_idx]
color_coefs = ['#e74c3c' if c > 0 else '#2ecc71' for c in top_coefs]
ax3.barh(top_names, top_coefs, color=color_coefs, edgecolor='black', alpha=0.8)
ax3.set_title(f'L2 Ridge (C={best_C}) — Top 20 Coefficients')
ax3.set_xlabel('Coefficient')
ax3.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('Data/05c_L2_coefs.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# === SAVING RESULTS ===
results = {
    'model': 'Logistic L2 (Ridge)',
    'best_params': str(grid.best_params_),
    'balanced_accuracy_mean': grid.best_score_,
    'balanced_accuracy_std': results_df.loc[grid.best_index_, 'std_test_score'],
}
pd.DataFrame([results]).to_csv('Data/results_L2.csv', index=False)
print("\n✅ Results saved: Data/results_L2.csv")



  # === 05d_logistic_elasticnet.py ===







  Logistic Regression ElasticNet with GridSearchCV



  Input: restaurants_train_pruned.csv



  Metric: balanced_accuracy





In [ ]:
# === PIPELINE ===
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        penalty='elasticnet',
        solver='saga',
        class_weight='balanced',
        max_iter=5000,
        random_state=42
    ))
])

# === GRID SEARCH ===
param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__l1_ratio': [0.25, 0.5, 0.75]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n⏳ GridSearchCV in progress...")
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='balanced_accuracy',
    refit=True,
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)
grid.fit(X, target)



In [ ]:
# === RESULTS ===
print(f"\n{'='*60}")
print(f"LOGISTIC REGRESSION ELASTICNET")
print(f"{'='*60}")
print(f"Best C: {grid.best_params_['model__C']}")
print(f"Best l1_ratio: {grid.best_params_['model__l1_ratio']}")
print(f"Best Balanced Accuracy (CV): {grid.best_score_:.4f}")

results_df = pd.DataFrame(grid.cv_results_)
for _, row in results_df.iterrows():
    c_val = row['param_model__C']
    l1 = row['param_model__l1_ratio']
    print(f"  C={c_val:<6} l1_ratio={l1:<5} → Train: {row['mean_train_score']:.4f}  Test: {row['mean_test_score']:.4f} (±{row['std_test_score']:.4f})")



In [ ]:
# === OOF PREDICTIONS ===
oof_preds = np.zeros(len(target))
oof_proba = np.zeros(len(target))

best_C = grid.best_params_['model__C']
best_l1 = grid.best_params_['model__l1_ratio']
for train_idx, val_idx in cv.split(X, target):
    pipe_fold = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            penalty='elasticnet', solver='saga', C=best_C, l1_ratio=best_l1,
            class_weight='balanced', max_iter=5000, random_state=42
        ))
    ])
    pipe_fold.fit(X.iloc[train_idx], target.iloc[train_idx])
    oof_preds[val_idx] = pipe_fold.predict(X.iloc[val_idx])
    oof_proba[val_idx] = pipe_fold.predict_proba(X.iloc[val_idx])[:, 1]

cm = confusion_matrix(target, oof_preds)
ba = balanced_accuracy_score(target, oof_preds)

print(f"\nConfusion Matrix (OOF):")
print(f"                 Predicted 0   Predicted 1")
print(f"  Actual 0 (open):   {cm[0,0]:>6d}        {cm[0,1]:>6d}")
print(f"  Actual 1 (closed): {cm[1,0]:>6d}        {cm[1,1]:>6d}")
print(f"\nBalanced Accuracy OOF: {ba:.4f}")
print(f"\nClassification Report (OOF):")
print(classification_report(target, oof_preds, target_names=['Open', 'Closed']))

best_model = grid.best_estimator_.named_steps['model']



In [ ]:
# === PLOT 1: Confusion Matrix ===
fig1, ax1 = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Open', 'Closed'],
            yticklabels=['Open', 'Closed'])
ax1.set_title(f'ElasticNet (C={best_C}, l1={best_l1}) — Confusion Matrix (OOF)')
ax1.set_ylabel('Actual')
ax1.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('Data/05d_EN_cm.png', dpi=150, bbox_inches='tight')
plt.show()

# === PLOT 2: Heatmap Balanced Accuracy (C × l1_ratio) ===
fig2, ax2 = plt.subplots(figsize=(8, 5))
pivot = results_df.pivot_table(
    values='mean_test_score',
    index='param_model__C',
    columns='param_model__l1_ratio'
)
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax2)
ax2.set_title('ElasticNet — Balanced Accuracy (C × l1_ratio)')
ax2.set_ylabel('C')
ax2.set_xlabel('l1_ratio')
plt.tight_layout()
plt.savefig('Data/05d_EN_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

# === PLOT 3: Top 20 Coefficients ===
fig3, ax3 = plt.subplots(figsize=(8, 6))
coef_vals = best_model.coef_[0]
top_idx = np.argsort(np.abs(coef_vals))[-20:]
top_names = [feature_names[i] for i in top_idx]
top_coefs = coef_vals[top_idx]
color_coefs = ['#e74c3c' if c > 0 else '#2ecc71' for c in top_coefs]
ax3.barh(top_names, top_coefs, color=color_coefs, edgecolor='black', alpha=0.8)
ax3.set_title(f'ElasticNet (C={best_C}, l1={best_l1}) — Top 20 Coefficients')
ax3.set_xlabel('Coefficient')
ax3.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('Data/05d_EN_coefs.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# === SAVING RESULTS ===
results = {
    'model': 'Logistic ElasticNet',
    'best_params': str(grid.best_params_),
    'balanced_accuracy_mean': grid.best_score_,
    'balanced_accuracy_std': results_df.loc[grid.best_index_, 'std_test_score'],
}
pd.DataFrame([results]).to_csv('Data/results_EN.csv', index=False)
print("\n✅ Results saved: Data/results_EN.csv")



  # === 05e_knn.py ===







  K-Nearest Neighbors with GridSearchCV



  Input: restaurants_train_pruned.csv



  Metric: balanced_accuracy

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# === PIPELINE ===
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier())
])

# === GRID SEARCH ===
# weights='distance' helps with imbalance:
# the nearest neighbors weigh more in the decision
param_grid = {
    'model__n_neighbors': [5, 11, 21, 31, 51],
    'model__weights': ['uniform', 'distance'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n⏳ GridSearchCV in progress...")
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='balanced_accuracy',
    refit=True,
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)
grid.fit(X, target)



In [ ]:
# === RESULTS ===
print(f"\n{'='*60}")
print(f"K-NEAREST NEIGHBORS")
print(f"{'='*60}")
print(f"Best n_neighbors: {grid.best_params_['model__n_neighbors']}")
print(f"Best weights: {grid.best_params_['model__weights']}")
print(f"Best Balanced Accuracy (CV): {grid.best_score_:.4f}")

results_df = pd.DataFrame(grid.cv_results_)
for _, row in results_df.iterrows():
    k = row['param_model__n_neighbors']
    w = row['param_model__weights']
    print(f"  k={k:<4} weights={w:<10} → Train: {row['mean_train_score']:.4f}  Test: {row['mean_test_score']:.4f} (±{row['std_test_score']:.4f})")



In [ ]:
# === OOF PREDICTIONS ===
oof_preds = np.zeros(len(target))

best_k = grid.best_params_['model__n_neighbors']
best_w = grid.best_params_['model__weights']
for train_idx, val_idx in cv.split(X, target):
    pipe_fold = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=best_k, weights=best_w))
    ])
    pipe_fold.fit(X.iloc[train_idx], target.iloc[train_idx])
    oof_preds[val_idx] = pipe_fold.predict(X.iloc[val_idx])

cm = confusion_matrix(target, oof_preds)
ba = balanced_accuracy_score(target, oof_preds)

print(f"\nConfusion Matrix (OOF):")
print(f"                 Predicted 0   Predicted 1")
print(f"  Actual 0 (open):   {cm[0,0]:>6d}        {cm[0,1]:>6d}")
print(f"  Actual 1 (closed): {cm[1,0]:>6d}        {cm[1,1]:>6d}")
print(f"\nBalanced Accuracy OOF: {ba:.4f}")
print(f"\nClassification Report (OOF):")
print(classification_report(target, oof_preds, target_names=['Open', 'Closed']))



In [ ]:
# === PLOT 1: Confusion Matrix ===
fig1, ax1 = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Open', 'Closed'],
            yticklabels=['Open', 'Closed'])
ax1.set_title(f'KNN (k={best_k}, {best_w}) — Confusion Matrix (OOF)')
ax1.set_ylabel('Actual')
ax1.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('Data/05e_KNN_cm.png', dpi=150, bbox_inches='tight')
plt.show()

# === PLOT 2: Balanced Accuracy vs K (per weight type) ===
fig2, ax2 = plt.subplots(figsize=(8, 5))
for w_type in ['uniform', 'distance']:
    mask = results_df['param_model__weights'] == w_type
    subset = results_df[mask].sort_values('param_model__n_neighbors')
    k_vals = subset['param_model__n_neighbors'].astype(int)
    ax2.plot(k_vals, subset['mean_test_score'], 'o-', label=f'Test ({w_type})')
    ax2.fill_between(k_vals,
                     subset['mean_test_score'] - subset['std_test_score'],
                     subset['mean_test_score'] + subset['std_test_score'],
                     alpha=0.15)
ax2.set_xlabel('n_neighbors (K)')
ax2.set_ylabel('Balanced Accuracy')
ax2.set_title('KNN — Balanced Accuracy vs K')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Data/05e_KNN_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

# === PLOT 3: Train vs Test (overfitting check) ===
fig3, ax3 = plt.subplots(figsize=(8, 5))
for w_type in ['uniform', 'distance']:
    mask = results_df['param_model__weights'] == w_type
    subset = results_df[mask].sort_values('param_model__n_neighbors')
    k_vals = subset['param_model__n_neighbors'].astype(int)
    ax3.plot(k_vals, subset['mean_train_score'], 's--', label=f'Train ({w_type})', alpha=0.7)
    ax3.plot(k_vals, subset['mean_test_score'], 'o-', label=f'Test ({w_type})')
ax3.set_xlabel('n_neighbors (K)')
ax3.set_ylabel('Balanced Accuracy')
ax3.set_title('KNN — Train vs Test (overfitting check)')
ax3.legend()
ax3.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Data/05e_KNN_overfit.png', dpi=150, bbox_inches='tight')
plt.show()




In [ ]:
# === SAVING RESULTS ===
results = {
    'model': 'KNN',
    'best_params': str(grid.best_params_),
    'balanced_accuracy_mean': grid.best_score_,
    'balanced_accuracy_std': results_df.loc[grid.best_index_, 'std_test_score'],
}
pd.DataFrame([results]).to_csv('Data/results_KNN.csv', index=False)
print("\n✅ Results saved: Data/results_KNN.csv")



  # === 05f_svc.py ===







  Support Vector Classifier (RBF kernel) with GridSearchCV



  Input: restaurants_train_pruned.csv



  Metric: balanced_accuracy







  NOTE: SVC is computationally heavy on 33k samples.



  Use probability=True to perform threshold analysis in model evaluation.

In [ ]:
from sklearn.svm import SVC

# === PIPELINE ===
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', SVC(
        kernel='rbf',
        class_weight='balanced',
        probability=False,
        random_state=42
    ))
])

# === GRID SEARCH ===
param_grid = {
    'model__C': [0.1, 1, 10],
    'model__gamma': ['scale', 'auto'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n⏳ GridSearchCV in progress (SVC is slow, be patient)...")
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='balanced_accuracy',
    refit=True,
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)
grid.fit(X, target)



In [ ]:
# === RESULTS ===
print(f"\n{'='*60}")
print(f"SUPPORT VECTOR CLASSIFIER (RBF)")
print(f"{'='*60}")
print(f"Best C: {grid.best_params_['model__C']}")
print(f"Best gamma: {grid.best_params_['model__gamma']}")
print(f"Best Balanced Accuracy (CV): {grid.best_score_:.4f}")



In [ ]:
results_df = pd.DataFrame(grid.cv_results_)
for _, row in results_df.iterrows():
    c_val = row['param_model__C']
    g = row['param_model__gamma']
    print(f"  C={c_val:<6} gamma={g:<8} → Train: {row['mean_train_score']:.4f}  Test: {row['mean_test_score']:.4f} (±{row['std_test_score']:.4f})")



In [ ]:
# === OOF PREDICTIONS ===
oof_preds = np.zeros(len(target))

best_C = grid.best_params_['model__C']
best_g = grid.best_params_['model__gamma']
for train_idx, val_idx in cv.split(X, target):
    pipe_fold = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', SVC(
            kernel='rbf', C=best_C, gamma=best_g,
            class_weight='balanced', probability=False, random_state=42
        ))
    ])
    pipe_fold.fit(X.iloc[train_idx], target.iloc[train_idx])
    oof_preds[val_idx] = pipe_fold.predict(X.iloc[val_idx])

cm = confusion_matrix(target, oof_preds)
ba = balanced_accuracy_score(target, oof_preds)

print(f"\nConfusion Matrix (OOF):")
print(f"                 Predicted 0   Predicted 1")
print(f"  Actual 0 (open):   {cm[0,0]:>6d}        {cm[0,1]:>6d}")
print(f"  Actual 1 (closed): {cm[1,0]:>6d}        {cm[1,1]:>6d}")
print(f"\nBalanced Accuracy OOF: {ba:.4f}")
print(f"\nClassification Report (OOF):")
print(classification_report(target, oof_preds, target_names=['Open', 'Closed']))



In [ ]:
# === PLOT 1: Confusion Matrix ===
fig1, ax1 = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Open', 'Closed'],
            yticklabels=['Open', 'Closed'])
ax1.set_title(f'SVC (C={best_C}, gamma={best_g}) — Confusion Matrix (OOF)')
ax1.set_ylabel('Actual')
ax1.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('Data/05f_SVC_cm.png', dpi=150, bbox_inches='tight')
plt.show()

# === PLOT 2: Heatmap Balanced Accuracy (C × gamma) ===
fig2, ax2 = plt.subplots(figsize=(8, 5))
pivot = results_df.pivot_table(
    values='mean_test_score',
    index='param_model__C',
    columns='param_model__gamma'
)
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax2)
ax2.set_title('SVC — Balanced Accuracy (C × gamma)')
ax2.set_ylabel('C')
ax2.set_xlabel('gamma')
plt.tight_layout()
plt.savefig('Data/05f_SVC_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

# === PLOT 3: Train vs Test ===
fig3, ax3 = plt.subplots(figsize=(8, 5))
for g_type in ['scale', 'auto']:
    mask = results_df['param_model__gamma'] == g_type
    subset = results_df[mask].sort_values('param_model__C')
    c_vals = subset['param_model__C'].astype(float)
    ax3.plot(c_vals, subset['mean_train_score'], 's--', label=f'Train (γ={g_type})', alpha=0.7)
    ax3.plot(c_vals, subset['mean_test_score'], 'o-', label=f'Test (γ={g_type})')
ax3.set_xscale('log')
ax3.set_xlabel('C')
ax3.set_ylabel('Balanced Accuracy')
ax3.set_title('SVC — Train vs Test (overfitting check)')
ax3.legend()
ax3.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Data/05f_SVC_overfit.png', dpi=150, bbox_inches='tight')
plt.show()




In [ ]:
# === SAVING RESULTS ===
results = {
    'model': 'SVC (RBF)',
    'best_params': str(grid.best_params_),
    'balanced_accuracy_mean': grid.best_score_,
    'balanced_accuracy_std': results_df.loc[grid.best_index_, 'std_test_score'],
}
pd.DataFrame([results]).to_csv('Data/results_SVC.csv', index=False)
print("\n✅ Results saved: Data/results_SVC.csv")



  # === 05g_scaler_penalty_search.py ===







  Systematic search: 3 Scalers × 3 Penalties × refined Cs



  Objective: find the combination that maximizes balanced_accuracy



  Input: restaurants_train_pruned.csv

In [ ]:
from sklearn.preprocessing import StandardScaler, PowerTransformer, RobustScaler
from sklearn.model_selection import GridSearchCV

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# === EXPERIMENTS CONFIGURATION ===
scalers = {
    'StandardScaler': StandardScaler(),
    'PowerTransformer': PowerTransformer(method='yeo-johnson'),
}

experiments = {
    'L1': {
        'penalty': 'l1',
        'solver': 'saga',
        'param_grid': {'model__C': [0.001, 0.008, 0.01, 0.05, 0.1, 1.0, 10.0]},
        'extra_params': {},
    },
    'L2': {
        'penalty': 'l2',
        'solver': 'lbfgs',
        'param_grid': {'model__C': [0.001, 0.008, 0.01, 0.05, 0.1, 1.0, 10.0]},
        'extra_params': {},
    },
    'ElasticNet': {
        'penalty': 'elasticnet',
        'solver': 'saga',
        'param_grid': {
            'model__C': [0.001, 0.008, 0.01, 0.05, 0.1, 1.0, 10.0],
            'model__l1_ratio': [0.5, 0.75, 0.9],
        },
        'extra_params': {},
    },
}



In [ ]:
# === EXECUTION ===
all_results = []
total = len(scalers) * len(experiments)
counter = 0

print(f"\n{'='*70}")
print(f"SYSTEMATIC SEARCH: {len(scalers)} Scalers × {len(experiments)} Penalties = {total} combinations")
print(f"{'='*70}\n")

for scaler_name, scaler in scalers.items():
    for exp_name, exp_config in experiments.items():
        counter += 1
        combo_name = f"{scaler_name} + {exp_name}"
        print(f"[{counter}/{total}] {combo_name}...", end=' ')
        
        pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', scaler.__class__(**scaler.get_params())),
            ('model', LogisticRegression(
                penalty=exp_config['penalty'],
                solver=exp_config['solver'],
                class_weight='balanced',
                max_iter=5000,
                random_state=42,
                **exp_config['extra_params'],
            ))
        ])
        
        grid = GridSearchCV(
            pipeline,
            exp_config['param_grid'],
            cv=cv,
            scoring='balanced_accuracy',
            refit=True,
            return_train_score=True,
            n_jobs=-1,
            verbose=0
        )
        grid.fit(X, target)
        
        best_train = grid.cv_results_['mean_train_score'][grid.best_index_]
        best_test = grid.best_score_
        best_std = grid.cv_results_['std_test_score'][grid.best_index_]
        
        print(f"BA={best_test:.4f} (±{best_std:.4f}) | Train={best_train:.4f} | {grid.best_params_}")
        
        all_results.append({
            'scaler': scaler_name,
            'penalty': exp_name,
            'combination': combo_name,
            'best_params': str(grid.best_params_),
            'ba_test_mean': best_test,
            'ba_test_std': best_std,
            'ba_train_mean': best_train,
            'overfit_gap': best_train - best_test,
        })



In [ ]:
# === RESULTS TABLE ===
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('ba_test_mean', ascending=False).reset_index(drop=True)

print(f"\n{'='*70}")
print(f"FINAL RANKING — ALL COMBINATIONS")
print(f"{'='*70}")
print(results_df[['combination', 'ba_test_mean', 'ba_test_std', 'ba_train_mean', 'overfit_gap', 'best_params']].to_string(index=True))



In [ ]:
# Winner
winner = results_df.iloc[0]
print(f"\n🏆 WINNER: {winner['combination']}")
print(f"   Balanced Accuracy: {winner['ba_test_mean']:.4f} (±{winner['ba_test_std']:.4f})")
print(f"   Parameters: {winner['best_params']}")
print(f"   Overfitting gap: {winner['overfit_gap']:.4f}")



In [ ]:
# === PLOT 1: Balanced Accuracy Ranking ===
fig1, ax1 = plt.subplots(figsize=(12, 7))
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(results_df))]
bars = ax1.barh(range(len(results_df)), results_df['ba_test_mean'],
                xerr=results_df['ba_test_std'],
                color=colors, edgecolor='black', alpha=0.85)
ax1.set_yticks(range(len(results_df)))
ax1.set_yticklabels(results_df['combination'], fontsize=10)
for i, (bar, val) in enumerate(zip(bars, results_df['ba_test_mean'])):
    ax1.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontweight='bold', fontsize=9)
ax1.set_xlabel('Balanced Accuracy (CV)', fontsize=12)
ax1.set_title('Scaler × Penalty Combinations Ranking', fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)
ax1.invert_yaxis()
plt.tight_layout()
plt.savefig('Data/05g_scaler_penalty_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

# === PLOT 2: Heatmap Scaler × Penalty ===
fig2, ax2 = plt.subplots(figsize=(8, 5))
pivot = results_df.pivot_table(
    values='ba_test_mean',
    index='scaler',
    columns='penalty'
)
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax2,
            linewidths=1, linecolor='white')
ax2.set_title('Balanced Accuracy — Scaler × Penalty', fontsize=14, fontweight='bold')
ax2.set_ylabel('Scaler')
ax2.set_xlabel('Penalty')
plt.tight_layout()
plt.savefig('Data/05g_scaler_penalty_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# === PLOT 3: Overfitting Gap ===
fig3, ax3 = plt.subplots(figsize=(12, 6))
colors_gap = ['#e74c3c' if g > 0.02 else '#f39c12' if g > 0.01 else '#2ecc71'
              for g in results_df['overfit_gap']]
ax3.barh(range(len(results_df)), results_df['overfit_gap'],
         color=colors_gap, edgecolor='black', alpha=0.85)
ax3.set_yticks(range(len(results_df)))
ax3.set_yticklabels(results_df['combination'], fontsize=10)
ax3.set_xlabel('Overfitting Gap (Train - Test)', fontsize=12)
ax3.set_title('Overfitting Gap per Combination', fontsize=14, fontweight='bold')
ax3.axvline(x=0.01, color='gray', linestyle=':', alpha=0.5, label='Gap = 0.01')
ax3.legend()
ax3.grid(axis='x', alpha=0.3)
ax3.invert_yaxis()
plt.tight_layout()
plt.savefig('Data/05g_overfitting_gap.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# === SAVING ===
results_df.to_csv('Data/05g_scaler_penalty_results.csv', index=False)
print("\n✅ Results saved: Data/05g_scaler_penalty_results.csv")



 # === 05h - Fine Grid of C x l1_ratio ===

In [ ]:
# === 05h — Fine grid C × l1_ratio ===

train = pd.read_csv('Data/restaurants_train_final.csv')
target = train['status_closed']
X = train.drop(columns=['restaurant_id', 'status_closed'])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

C_values = [0.03, 0.05, 0.07, 0.1, 0.15, 0.2, 0.3]
l1_values = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

results = []
for C in C_values:
    for l1 in l1_values:
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', PowerTransformer(method='yeo-johnson')),
            ('model', LogisticRegression(
                penalty='elasticnet', solver='saga', C=C,
                l1_ratio=l1, class_weight='balanced',
                max_iter=5000, random_state=42
            ))
        ])
        scores = cross_val_score(pipe, X, target, cv=cv,
                                 scoring='balanced_accuracy', n_jobs=-1)
        results.append({
            'C': C, 'l1_ratio': l1,
            'ba_mean': scores.mean(), 'ba_std': scores.std()
        })
        print(f"C={C:<5} l1={l1:<4} → BA = {scores.mean():.4f} (±{scores.std():.4f})")

results_df = pd.DataFrame(results).sort_values('ba_mean', ascending=False)
print("\n" + "=" * 60)
print("TOP 10 COMBINATIONS")
print("=" * 60)
print(results_df.head(10).to_string(index=False))
print(f"\nCurrent (C=0.1, l1=0.5): BA = 0.6774")
print(f"Best found: C={results_df.iloc[0]['C']}, l1={results_df.iloc[0]['l1_ratio']} → BA = {results_df.iloc[0]['ba_mean']:.4f}")


In [ ]:
# === Test B — All post-FE2 features (without pruning/LASSO) ===

train = pd.read_csv('Data/restaurants_train_fe2.csv')
target = train['status_closed']
X = train.drop(columns=['restaurant_id', 'status_closed'])

print(f"Feature: {X.shape[1]} | Samples: {X.shape[0]}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', PowerTransformer(method='yeo-johnson')),
    ('model', LogisticRegression(
        penalty='elasticnet', solver='saga', C=0.1,
        l1_ratio=0.5, class_weight='balanced',
        max_iter=5000, random_state=42
    ))
])

scores = cross_val_score(pipe, X, target, cv=cv,
                         scoring='balanced_accuracy', n_jobs=-1)

print(f"\nFold scores: {scores}")
print(f"Mean BA: {scores.mean():.4f} (±{scores.std():.4f})")
print(f"\nComparison: 84 features → 0.6774 | {X.shape[1]} features → {scores.mean():.4f}")


In [ ]:
# === SVC Test with PowerTransformer — GridSearchCV ===

from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV

train = pd.read_csv('Data/restaurants_train_final.csv')
target = train['status_closed']
X = train.drop(columns=['restaurant_id', 'status_closed'])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', PowerTransformer(method='yeo-johnson')),
    ('model', SVC(kernel='rbf', class_weight='balanced', random_state=42))
])

param_grid = {'model__C': [0.01, 0.1, 1, 10, 100]}

grid = GridSearchCV(pipe, param_grid, cv=cv, scoring='balanced_accuracy',
                    n_jobs=-1, verbose=1)
grid.fit(X, target)

print(f"\nBest C: {grid.best_params_['model__C']}")
print(f"Best BA: {grid.best_score_:.4f}")
print(f"\nAll results:")
results = pd.DataFrame(grid.cv_results_)
for _, row in results.iterrows():
    C = row['param_model__C']
    print(f"  C={C:<6} → BA = {row['mean_test_score']:.4f} (±{row['std_test_score']:.4f})")

print(f"\nLogReg ElasticNet Reference: BA = 0.6774")


  # === 06_model_comparison.py ===







  Comparison of all tested models



  Input: results_*.csv saved by each 05* script



  Output: comparative plots

In [ ]:
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')



In [ ]:
results = pd.DataFrame([
    # --- Old results (pre-advanced FE, 105 features) ---
    {'model': 'Baseline Logistic (old)', 'balanced_accuracy_mean': 0.6566, 'balanced_accuracy_std': 0.0103, 'features_used': 105},
    {'model': 'PowerTransformer + ElasticNet (old)', 'balanced_accuracy_mean': 0.6695, 'balanced_accuracy_std': 0.0083, 'features_used': 105},
    {'model': 'KNN (old)', 'balanced_accuracy_mean': 0.5224, 'balanced_accuracy_std': 0.0041, 'features_used': 105},
    {'model': 'SVC RBF (old)', 'balanced_accuracy_mean': 0.6522, 'balanced_accuracy_std': 0.0056, 'features_used': 105},
    
    # --- Results with WoE batch 2 (post-LASSO, 106 features) ---
    {'model': 'Pure Logistic (no penalty)', 'balanced_accuracy_mean': 0.6786, 'balanced_accuracy_std': 0.0046, 'features_used': 106},
    {'model': 'LASSO (L1)', 'balanced_accuracy_mean': 0.6784, 'balanced_accuracy_std': 0.0042, 'features_used': 106},
    {'model': 'Ridge (L2)', 'balanced_accuracy_mean': 0.6787, 'balanced_accuracy_std': 0.0044, 'features_used': 106},
    {'model': 'ElasticNet (C=0.1, l1=0.75)', 'balanced_accuracy_mean': 0.6791, 'balanced_accuracy_std': 0.0028, 'features_used': 106},
    {'model': 'KNN (k=5)', 'balanced_accuracy_mean': 0.5175, 'balanced_accuracy_std': 0.0039, 'features_used': 106},
    {'model': 'SVC RBF (C=0.1)', 'balanced_accuracy_mean': 0.6760, 'balanced_accuracy_std': 0.0081, 'features_used': 106},
    
    # --- Systematic search Scaler × Penalty ---
    {'model': 'StandardScaler + L1', 'balanced_accuracy_mean': 0.6784, 'balanced_accuracy_std': 0.0042, 'features_used': 106},
    {'model': 'StandardScaler + L2', 'balanced_accuracy_mean': 0.6787, 'balanced_accuracy_std': 0.0044, 'features_used': 106},
    {'model': 'StandardScaler + ElasticNet', 'balanced_accuracy_mean': 0.6791, 'balanced_accuracy_std': 0.0028, 'features_used': 106},
    {'model': 'PowerTransformer + L1', 'balanced_accuracy_mean': 0.6838, 'balanced_accuracy_std': 0.0051, 'features_used': 106},
    {'model': 'PowerTransformer + ElasticNet', 'balanced_accuracy_mean': 0.6849, 'balanced_accuracy_std': 0.0049, 'features_used': 106},
    {'model': 'PowerTransformer + L2 ★', 'balanced_accuracy_mean': 0.6852, 'balanced_accuracy_std': 0.0036, 'features_used': 106},
])

results = results.sort_values('balanced_accuracy_mean', ascending=False).reset_index(drop=True)

print("=" * 70)
print("MODELS COMPARISON — BALANCED ACCURACY")
print("=" * 70)
print(results[['model', 'balanced_accuracy_mean', 'balanced_accuracy_std', 'features_used']].to_string(index=False))

print(f"\n→ Total improvement: {0.6852 - 0.6566:.4f} (+{(0.6852-0.6566)*100:.2f} pp)")
print(f"→ Best model: PowerTransformer + L2 (C=0.1)")
print(f"→ Previous Kaggle: 0.686 | Current CV: 0.6852")


In [ ]:
# === PLOT 1: Comparative Balanced Accuracy ===
fig1, ax1 = plt.subplots(figsize=(10, 6))
colors = ['#2ecc71' if i == 0 else '#3498db' if ba > 0.65 else '#e74c3c'
          for i, ba in enumerate(results['balanced_accuracy_mean'])]
bars = ax1.barh(results['model'], results['balanced_accuracy_mean'],
                xerr=results['balanced_accuracy_std'],
                color=colors, edgecolor='black', alpha=0.85)

for bar, val in zip(bars, results['balanced_accuracy_mean']):
    ax1.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontweight='bold')

ax1.set_xlabel('Balanced Accuracy (CV)')
ax1.set_title('Models Comparison — Balanced Accuracy', fontsize=13, fontweight='bold')
ax1.set_xlim(0.45, 0.72)
ax1.axvline(x=0.5, color='gray', linestyle=':', alpha=0.5, label='Random (0.50)')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('Data/06_comparison_balanced_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

# === PLOT 2: Stability (std of balanced accuracy) ===
fig2, ax2 = plt.subplots(figsize=(10, 5))
ax2.bar(results['model'], results['balanced_accuracy_std'],
        color='#e67e22', edgecolor='black', alpha=0.8)
ax2.set_ylabel('Std Balanced Accuracy (CV)')
ax2.set_title('Models Stability — Standard Deviation across Folds', fontsize=13, fontweight='bold')
ax2.set_xticklabels(results['model'], rotation=20, ha='right')
ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('Data/06_comparison_stability.png', dpi=150, bbox_inches='tight')
plt.show()

# === SUMMARY TABLE ===
print("\n" + "=" * 70)
print("SUMMARY TABLE")
print("=" * 70)
print(results.to_string(index=False))

print("\n🏆 BEST MODEL: PowerTransformer + L2 (C=1.0)")
print("   Balanced Accuracy: 0.6852 (±0.0036)")
print("   Features used: 106 (post LASSO selection)")
print("   Previous Kaggle: 0.686 → expected improvement: +1.4 pp")

results.to_csv('Data/06_model_comparison.csv', index=False)
print("\n✅ Table saved: Data/06_model_comparison.csv")



  # === 07_model_evaluation.py ===







  In-depth Evaluation — Logistic L1 (LASSO), best model



  Includes: ROC, Precision-Recall, threshold analysis, error costs



  Input: restaurants_train_pruned.csv



  Output: plots + optimal threshold

In [ ]:

from sklearn.metrics import (confusion_matrix, balanced_accuracy_score,
                             classification_report, roc_curve, auc,
                             precision_recall_curve, average_precision_score)
import warnings
warnings.filterwarnings('ignore')

# === LOADING ===
train = pd.read_csv('Data/restaurants_train_final.csv')
target = train['status_closed']
X = train.drop(columns=['restaurant_id', 'status_closed'])
feature_names = X.columns.tolist()
print(f"Feature: {X.shape[1]} | Samples: {X.shape[0]}")

# === OOF PREDICTIONS WITH PROBABILITY ===
# Use the best model: RIDGE with C=0.1 and PowerTransformer
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_proba = np.zeros(len(target))

print("⏳ Generating OOF predictions...")
for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X, target)):
    pipe_fold = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', PowerTransformer(method='yeo-johnson')),
        ('model', LogisticRegression(
            penalty='elasticnet', solver='saga', C=0.1, l1_ratio=0.2,
            class_weight='balanced', max_iter=5000, random_state=42
        ))
    ])
    pipe_fold.fit(X.iloc[train_idx], target.iloc[train_idx])
    oof_proba[val_idx] = pipe_fold.predict_proba(X.iloc[val_idx])[:, 1]
    print(f"  Fold {fold_idx+1} completed")

# Predictions with default threshold 0.5
oof_preds_default = (oof_proba >= 0.5).astype(int)
ba_default = balanced_accuracy_score(target, oof_preds_default)
cm_default = confusion_matrix(target, oof_preds_default)

print(f"\n{'='*60}")
print(f"DEFAULT THRESHOLD (0.5)")
print(f"{'='*60}")
print(f"Balanced Accuracy: {ba_default:.4f}")
print(f"Confusion Matrix:")
print(f"  Actual 0 (open):   {cm_default[0,0]:>6d} TN  |  {cm_default[0,1]:>6d} FP")
print(f"  Actual 1 (closed): {cm_default[1,0]:>6d} FN  |  {cm_default[1,1]:>6d} TP")



In [ ]:
# =====================================================================
# THRESHOLD ANALYSIS — Look for the threshold that maximizes balanced acc
# =====================================================================
thresholds = np.arange(0.05, 0.95, 0.005)
ba_scores = []
recall_0_scores = []
recall_1_scores = []
precision_1_scores = []
f1_1_scores = []

for t in thresholds:
    preds_t = (oof_proba >= t).astype(int)
    cm_t = confusion_matrix(target, preds_t)
    
    # Class 0 recall (specificity)
    tn, fp, fn, tp = cm_t.ravel()
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1_1 = 2 * precision_1 * recall_1 / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0
    
    ba = (recall_0 + recall_1) / 2
    
    ba_scores.append(ba)
    recall_0_scores.append(recall_0)
    recall_1_scores.append(recall_1)
    precision_1_scores.append(precision_1)
    f1_1_scores.append(f1_1)

ba_scores = np.array(ba_scores)
best_idx = np.argmax(ba_scores)
best_threshold = thresholds[best_idx]
best_ba = ba_scores[best_idx]

print(f"\n{'='*60}")
print(f"OPTIMAL THRESHOLD FOR BALANCED ACCURACY")
print(f"{'='*60}")
print(f"Optimal threshold: {best_threshold:.3f}")
print(f"Balanced Accuracy:  {best_ba:.4f}")
print(f"Improvement vs 0.5: +{(best_ba - ba_default)*100:.2f} percentage points")



In [ ]:
# Confusion matrix with optimal threshold
oof_preds_best = (oof_proba >= best_threshold).astype(int)
cm_best = confusion_matrix(target, oof_preds_best)
tn, fp, fn, tp = cm_best.ravel()

print(f"\nConfusion Matrix (threshold={best_threshold:.3f}):")
print(f"  Actual 0 (open):   {tn:>6d} TN  |  {fp:>6d} FP")
print(f"  Actual 1 (closed): {fn:>6d} FN  |  {tp:>6d} TP")
print(f"\nClassification Report (threshold={best_threshold:.3f}):")
print(classification_report(target, oof_preds_best, target_names=['Open', 'Closed']))



In [ ]:
# === ERROR COSTS ANALYSIS ===
print(f"{'='*60}")
print(f"ERROR ANALYSIS — TYPE 1 vs TYPE 2")
print(f"{'='*60}")
print(f"\n  With DEFAULT threshold (0.5):")
print(f"    FP (Type I  — predicts Closed, is Open):   {cm_default[0,1]:>6d}  ({cm_default[0,1]/len(target)*100:.1f}%)")
print(f"    FN (Type II — predicts Open, is Closed):    {cm_default[1,0]:>6d}  ({cm_default[1,0]/len(target)*100:.1f}%)")
print(f"\n  With OPTIMAL threshold ({best_threshold:.3f}):")
print(f"    FP (Type I  — predicts Closed, is Open):   {fp:>6d}  ({fp/len(target)*100:.1f}%)")
print(f"    FN (Type II — predicts Open, is Closed):    {fn:>6d}  ({fn/len(target)*100:.1f}%)")
print(f"\n  Trade-off:")
print(f"    Lowering the threshold → more restaurants predicted as Closed")
print(f"    → increases Recall(Closed) but also increases False Positives")
print(f"    → the optimal threshold balances the two errors to maximize BA")



In [ ]:
# =====================================================================
# PLOT 1: THRESHOLD vs BALANCED ACCURACY (with optimal point)
# =====================================================================
fig1, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(thresholds, ba_scores, color='#2c3e50', linewidth=2, label='Balanced Accuracy')
ax1.axvline(x=0.5, color='gray', linestyle=':', alpha=0.7, label=f'Default (0.5) → BA={ba_default:.4f}')
ax1.axvline(x=best_threshold, color='#e74c3c', linestyle='--', linewidth=2,
            label=f'Optimal ({best_threshold:.3f}) → BA={best_ba:.4f}')
ax1.scatter([best_threshold], [best_ba], color='#e74c3c', s=150, zorder=5, edgecolors='black')
ax1.set_xlabel('Threshold', fontsize=12)
ax1.set_ylabel('Balanced Accuracy', fontsize=12)
ax1.set_title('Optimal Threshold for Balanced Accuracy', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0.05, 0.95)
plt.tight_layout()
plt.savefig('Data/07_threshold_balanced_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

# =====================================================================
# PLOT 2: METRICS vs THRESHOLD
# =====================================================================
fig2, ax2 = plt.subplots(figsize=(10, 6))
ax2.plot(thresholds, recall_1_scores, label='Recall (Closed)', color='#e74c3c', linewidth=2)
ax2.plot(thresholds, recall_0_scores, label='Specificity (Open)', color='#2ecc71', linewidth=2)
ax2.plot(thresholds, precision_1_scores, label='Precision (Closed)', color='#3498db', linewidth=2)
ax2.plot(thresholds, f1_1_scores, label='F1 (Closed)', color='#9b59b6', linewidth=2)
ax2.plot(thresholds, ba_scores, label='Balanced Accuracy', color='#2c3e50', linewidth=2, linestyle='--')
ax2.axvline(x=best_threshold, color='#e74c3c', linestyle=':', alpha=0.7,
            label=f'Optimal threshold ({best_threshold:.3f})')
ax2.set_xlabel('Threshold', fontsize=12)
ax2.set_ylabel('Score', fontsize=12)
ax2.set_title('All Metrics vs Threshold', fontsize=14, fontweight='bold')
ax2.legend(loc='center left', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0.05, 0.95)
plt.tight_layout()
plt.savefig('Data/07_metrics_vs_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

# =====================================================================
# PLOT 3: ROC CURVE
# =====================================================================
fpr, tpr, roc_thresholds = roc_curve(target, oof_proba)
roc_auc = auc(fpr, tpr)

fig3, ax3 = plt.subplots(figsize=(7, 7))
ax3.plot(fpr, tpr, color='#2c3e50', linewidth=2, label=f'ROC (AUC = {roc_auc:.4f})')
ax3.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC = 0.5)')
ax3.set_xlabel('False Positive Rate', fontsize=12)
ax3.set_ylabel('True Positive Rate', fontsize=12)
ax3.set_title('ROC Curve — PowerTransformer + L2 (C=0.1)', fontsize=14, fontweight='bold')
ax3.legend(fontsize=11)
ax3.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Data/07_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

# =====================================================================
# PLOT 4: PRECISION-RECALL CURVE
# =====================================================================
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(target, oof_proba)
ap = average_precision_score(target, oof_proba)

fig4, ax4 = plt.subplots(figsize=(7, 7))
ax4.plot(recall_curve, precision_curve, color='#e67e22', linewidth=2,
         label=f'PR Curve (AP = {ap:.4f})')
ax4.axhline(y=target.mean(), color='gray', linestyle=':', alpha=0.7,
            label=f'Baseline (prevalence = {target.mean():.3f})')
ax4.set_xlabel('Recall', fontsize=12)
ax4.set_ylabel('Precision', fontsize=12)
ax4.set_title('Precision-Recall Curve — PowerTransformer + L2 (C=0.1)', fontsize=14, fontweight='bold')
ax4.legend(fontsize=11)
ax4.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Data/07_precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.show()

# =====================================================================
# PLOT 5: CONFUSION MATRIX COMPARISON (default vs optimal)
# =====================================================================
fig5, (ax5a, ax5b) = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm_default, annot=True, fmt='d', cmap='Blues', ax=ax5a,
            xticklabels=['Open', 'Closed'], yticklabels=['Open', 'Closed'])
ax5a.set_title(f'Threshold = 0.5\nBA = {ba_default:.4f}', fontsize=12)
ax5a.set_ylabel('Actual')
ax5a.set_xlabel('Predicted')

sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues', ax=ax5b,
            xticklabels=['Open', 'Closed'], yticklabels=['Open', 'Closed'])
ax5b.set_title(f'Threshold = {best_threshold:.3f}\nBA = {best_ba:.4f}', fontsize=12)
ax5b.set_ylabel('Actual')
ax5b.set_xlabel('Predicted')

fig5.suptitle('Confusion Matrix Comparison — Default vs Optimal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('Data/07_confusion_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# =====================================================================
# PLOT 6: PROBABILITY DISTRIBUTION BY CLASS
# =====================================================================
fig6, ax6 = plt.subplots(figsize=(10, 5))
ax6.hist(oof_proba[target == 0], bins=50, alpha=0.6, color='#2ecc71', label='Open (0)', density=True)
ax6.hist(oof_proba[target == 1], bins=50, alpha=0.6, color='#e74c3c', label='Closed (1)', density=True)
ax6.axvline(x=0.5, color='gray', linestyle=':', linewidth=2, label='Threshold 0.5')
ax6.axvline(x=best_threshold, color='black', linestyle='--', linewidth=2,
            label=f'Optimal threshold ({best_threshold:.3f})')
ax6.set_xlabel('Predicted Probability P(Closed)', fontsize=12)
ax6.set_ylabel('Density', fontsize=12)
ax6.set_title('Probability Distribution by Class', fontsize=14, fontweight='bold')
ax6.legend(fontsize=11)
ax6.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Data/07_probability_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# === FINAL SUMMARY ===
print(f"\n{'='*60}")
print(f"EVALUATION SUMMARY")
print(f"{'='*60}")
print(f"Model: PowerTransformer + L2 (C=0.1)")
print(f"ROC AUC: {roc_auc:.4f}")
print(f"Average Precision: {ap:.4f}")
print(f"\nDefault threshold (0.5):   BA = {ba_default:.4f}")
print(f"Optimal threshold ({best_threshold:.3f}): BA = {best_ba:.4f}")
print(f"Improvement: +{(best_ba - ba_default)*100:.2f} pp")
print(f"\n→ Use threshold = {best_threshold:.3f} for the Kaggle submission")



In [ ]:
# Save the optimal threshold for the submission script
eval_results = {
    'model': 'PowerTransformer + L2 (C=0.1)',
    'C': 0.1,
    'l1_ratio': None,
    'threshold_default': 0.5,
    'ba_default': ba_default,
    'threshold_optimal': best_threshold,
    'ba_optimal': best_ba,
    'roc_auc': roc_auc,
    'average_precision': ap,
}
pd.DataFrame([eval_results]).to_csv('Data/07_evaluation_results.csv', index=False)
print("\n✅ Results saved: Data/07_evaluation_results.csv")



  # === 08_submission.py ===







  Kaggle Submission — PowerTransformer + ElasticNet (C=0.1, l1_ratio=0.5)



  Input: restaurants_train_final.csv, restaurants_test_final.csv



  Output: submission.csv

In [ ]:
# === LOADING ===
train = pd.read_csv('Data/restaurants_train_final.csv')
test  = pd.read_csv('Data/restaurants_test_final.csv')

target = train['status_closed']
X_train = train.drop(columns=['restaurant_id', 'status_closed'])
X_test  = test.drop(columns=['restaurant_id'])

print(f"Train: {X_train.shape}")
print(f"Test:  {X_test.shape}")



In [ ]:
# === FINAL PIPELINE ===
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', PowerTransformer(method='yeo-johnson')),
    ('model', LogisticRegression(
        penalty='elasticnet',
        solver='saga',
        C=0.1,
        l1_ratio=0.2,
        class_weight='balanced',
        max_iter=5000,
        random_state=42
    ))
])

# === FIT ON ALL TRAIN DATA ===
print("\n⏳ Training on the complete dataset...")
pipeline.fit(X_train, target)
print("✅ Training completed")




In [ ]:
# === PREDICTIONS ===
THRESHOLD = 0.470  # Use the optimal threshold found in the analysis

test_proba = pipeline.predict_proba(X_test)[:, 1]
test_preds = (test_proba >= THRESHOLD).astype(int)

print(f"\nThreshold: {THRESHOLD}")
print(f"Predictions on the test set:")
print(f"  Class 0 (Open):   {(test_preds == 0).sum()}")
print(f"  Class 1 (Closed): {(test_preds == 1).sum()}")
print(f"  % Closed: {test_preds.mean()*100:.1f}%")


In [ ]:
# === CREATION OF SUBMISSION FILE ===
submission = pd.DataFrame({
    'restaurant_id': test['restaurant_id'],
    'status_closed': test_preds
})

# Check format
print(f"\nSubmission shape: {submission.shape}")
print(f"Columns: {submission.columns.tolist()}")
print(f"First rows:")
print(submission.head(10))

# Comparison with sample submission
sample = pd.read_csv('Data/restaurant_sample_submission.csv')
print(f"\nSample submission shape: {sample.shape}")
assert submission.shape == sample.shape, "⚠️ Shape different from sample!"
assert list(submission.columns) == list(sample.columns), "⚠️ Columns different from sample!"
print("✅ Format complies with sample submission")



In [ ]:
# === SAVING ===
submission.to_csv('Data/submission.csv', index=False)
print("\n✅ File saved: Data/submission.csv")
print("\n📤 Upload Data/submission.csv to Kaggle to get the score!")